# Predicción de Popularidad Musical en Spotify
## Notebook 3 (EV3): Optimización, Ensambles, Generalización y Selección de la Solución Óptima

**Asignatura:** MLY1101 Machine Learning · Duoc UC — **Grupo 8 · Autor:** Johan Soto
**Caso C:** Inteligencia musical y predicción de popularidad de canciones (Spotify Tracks)

### Resumen
Este notebook cierra el ciclo CRISP-DM iniciado en **EV1 (EDA y limpieza)** y **EV2 corregida (modelamiento)**.
Usa exactamente el mismo dataset, la misma definición de éxito (`popularity ≥ 50`), las mismas variables
(**set de lanzamiento**: audio + ingeniería + género, sin Target Encoding), la misma
**partición por artista** (`GroupShuffleSplit`, `random_state=42`: ningún artista en train y test) y el mismo
balanceo (`class_weight` / `scale_pos_weight`). El código de carga y partición es literalmente el de EV2.
Sobre esa base, EV3 agrega lo que exige la pauta:

| Indicador de la pauta | Peso | Dónde se evidencia en este notebook |
|---|---|---|
| **IE9** Optimización de hiperparámetros | 20% | Sección 5 (RandomizedSearchCV, Δ base vs optimizado) y Sección 9 (umbral) |
| **IE10** Técnicas de ensamble | 30% | Secciones 5–6 (Bagging = RF, Boosting = XGB, Votación suave RF+XGB) y su aporte frente al árbol simple |
| **IE11** Capacidad de generalización | 20% | Secciones 8, 11, 12 (CV K=5 agrupada por artista, comparación con CV aleatoria, Train vs CV, curvas de aprendizaje) |
| **IE12** Selección y justificación de la solución óptima | 30% | Sección 14 (matriz de decisión, sensibilidad, párrafo ejecutivo, KPIs) y Sección 16 (resumen para la defensa) |

### Estructura (sigue la lógica de la ActividadML de referencia)
1. Contexto, continuidad con EV1/EV2 y KPIs
2. Configuración del entorno
3. Carga, verificación, ingeniería de características y partición por artista (idéntico a EV2)
4. Pipeline reproducible sin fuga de información
5. Optimización de hiperparámetros (IE9)
6. Ensambles y entrenamiento de los modelos finales (IE10)
7. **Comparación lado a lado (métricas, Δ, FN/FP)** y **matrices de confusión comparadas**
8. Motor de validación cruzada K=5 agrupada por artista (probabilidades fuera de pliegue)
9. **Curvas ROC y Precision-Recall comparadas** y **umbral óptimo con código (maximizar Recall bajo restricción)**
10. **Feature Importance con código (RF + XGB + permutation)** y ablación "música vs. contexto"
11. **Auditoría estadística con validación cruzada K=5** (+ cuánto inflaba la validación aleatoria)
12. **Diagnóstico de sobreajuste (Train vs. CV)**
13. Análisis de sesgos y equidad con evidencia cuantitativa (géneros, artistas emergentes, segmentos sonoros)
14. **Sustentación de la solución óptima** (matriz de decisión + verificación de KPIs)
15. Conclusiones, limitaciones y recomendaciones
16. **Descubrimientos, insights y resumen para la defensa** (por qué este modelo, qué significan las métricas,
    sesgos y qué hicimos, impacto y precauciones)
17. Exportación de artefactos

> **Tiempo de ejecución estimado:** 40–60 min la primera vez en una CPU estándar (Random Forest es el paso más lento).
> Cada paso pesado se guarda en `models/cache_ev3/`: **si el entorno se reinicia, vuelva a ejecutar todo y el notebook
> retoma desde el último paso guardado** (los pasos ya calculados se cargan en segundos). En Google Colab, monte
> Google Drive y ejecute desde una carpeta de Drive para que los guardados no se pierdan al reiniciar.
> Prueba rápida: `MODO_RAPIDO = True` en la Sección 2.

## 1. Contexto, continuidad con EV1/EV2 y KPIs

### Problema de negocio
Los sellos discográficos deben decidir **antes del lanzamiento** en qué canciones invertir presupuesto
promocional. El objetivo es estimar la probabilidad de que una canción sea un **hit** (`popularity ≥ 50`,
≈ percentil 75; prevalencia ≈ 23.7%) para (a) no perder canciones exitosas y (b) no malgastar presupuesto.
Escenario evaluado: **artistas que el modelo nunca vio** (el caso real y más exigente).

### Qué heredamos de EV2 (corregida) y qué agrega EV3
| Elemento | EV2 corregida | EV3 | Por qué |
|---|---|---|---|
| Dataset / target / variables | `spotify_tracks_clean.csv`, hit ≥ 50, set de lanzamiento sin TE | **Igual** (mismo código) | Coherencia entre entregas |
| Partición | `GroupShuffleSplit` por artista principal | **Igual** → mismo test | Evita fuga por artista, álbum y canciones repetidas con otro `track_id` |
| Target Encoding de artista/álbum | Eliminado del modelo final tras la ablación | Solo como variante de contraste en la ablación (10.3) | El álbum filtra información (compilados) y no existe al lanzar; el artista no aporta con artistas nuevos |
| Balanceo | `class_weight` / `scale_pos_weight` | **Igual** | SMOTE generaba filas inválidas en el one-hot de género |
| Hiperparámetros | Fijados a mano | **RandomizedSearchCV** con CV agrupada y criterio PR-AUC | IE9 |
| Umbral | Máximo F2 en validación | **Recall ≥ 70% con el mínimo Waste Rate**, elegido con probabilidades fuera de pliegue (OOF) | Cumple el KPI Recall@Hit gastando lo mínimo. En la ActividadML el umbral se elegía mirando el test; aquí el test se usa **una sola vez** |
| Ensambles | RF y XGB por separado | RF (bagging), XGB (boosting) y **Votación suave**, contra un **árbol simple** | IE10 |
| Generalización | Un hold-out | **CV K=5 agrupada por artista**, comparación con CV aleatoria, Train vs CV, curvas de aprendizaje | IE11 |
| Selección | Mejor PR-AUC en validación | **Matriz de decisión ponderada sobre CV** + sensibilidad de pesos | IE12 |

### KPIs del proyecto
Tabla única cargada desde `kpis_spotify.csv` (la misma de EV1 y EV2) y verificada completa en la Sección 14.4.

## 2. Configuración del entorno

Se fijan semillas y constantes de negocio en un solo lugar para que todo el notebook sea reproducible.
`RECALL_MIN` y `WASTE_MAX` provienen directamente de las metas de `kpis_spotify.csv`
("Detectar al menos el 70% de las canciones exitosas" y "No malgastar más del 15% del presupuesto").

**Requisitos:** scikit-learn ≥ 1.4 (TargetEncoder estable), xgboost ≥ 2.1.4 (compatibilidad con scikit-learn ≥ 1.6).

In [ ]:
import os, time, textwrap, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import sklearn
import xgboost
from scipy.stats import randint, uniform, loguniform

from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (train_test_split, GroupShuffleSplit, StratifiedKFold, StratifiedGroupKFold,
                                     RandomizedSearchCV, cross_val_score, cross_validate, learning_curve)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
                             roc_auc_score, average_precision_score, brier_score_loss,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve,
                             precision_recall_curve, ndcg_score,
                             mean_absolute_error, mean_squared_error, r2_score)
from xgboost import XGBClassifier, XGBRegressor

# ---------------- Constantes del proyecto ----------------
RANDOM_STATE = 42
UMBRAL_HIT   = 50      # definición de éxito (igual que EV2)
RECALL_MIN   = 0.70    # KPI Recall@Hit (kpis_spotify.csv)
WASTE_MAX    = 0.15    # KPI Waste Rate (kpis_spotify.csv)
K_FOLDS      = 5       # auditoría estadística exigida

MODO_RAPIDO  = False   # True = prueba rápida (menos iteraciones y árboles)
USAR_CACHE   = True    # guarda cada paso pesado en models/cache_ev3/: si el entorno se cae, re-ejecutar retoma
BORRAR_CACHE = False   # True = recalcula todo desde cero (usar si se modifica el código o los datos)

N_ITER_RF    = 4    if MODO_RAPIDO else 10
N_ITER_XGB   = 8    if MODO_RAPIDO else 20
SEARCH_FRAC  = 0.20 if MODO_RAPIDO else 0.30   # fracción de train usada en la búsqueda
N_ARBOLES_RF_BASE = 200 if MODO_RAPIDO else 500

np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titleweight': 'bold', 'figure.dpi': 100})

# Paleta categórica fija: cada modelo conserva SIEMPRE su color en todos los gráficos
COLORES = {
    'Árbol de Decisión':   '#8c8c8c',
    'Regresión Logística': '#eda100',
    'Random Forest':       '#1baf7a',
    'XGBoost':             '#2a78d6',
    'Votación RF+XGB':     '#eb6834',
}

# ---------------- Rutas (estructura profesional del proyecto) ----------------
ROOT = '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'
DIRS = {k: os.path.join(ROOT, k) for k in ['data', 'models', 'images', 'reports']}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

def ubicar(nombre):
    '''Busca el archivo en ../data (estructura del repositorio), data/, carpeta actual o Colab.'''
    for base in ['../data', 'data', DIRS['data'], '.', '/content', '/content/drive/MyDrive']:
        p = os.path.join(base, nombre)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"No se encontró {nombre}. Colóquelo en la carpeta data/ del proyecto.")

CACHE_DIR = os.path.join(DIRS['models'], 'cache_ev3')
os.makedirs(CACHE_DIR, exist_ok=True)
if BORRAR_CACHE:
    for f in os.listdir(CACHE_DIR):
        os.remove(os.path.join(CACHE_DIR, f))

def cacheado(nombre, funcion):
    '''Punto de guardado: si el resultado ya existe en disco lo carga; si no, lo calcula y lo guarda.
    Permite retomar la ejecución si el entorno se reinicia (límite de tiempo de Jupyter/Colab).'''
    ruta = os.path.join(CACHE_DIR, f"{nombre}_{'rapido' if MODO_RAPIDO else 'completo'}.joblib")
    if USAR_CACHE and os.path.exists(ruta):
        print(f"[guardado] '{nombre}' cargado desde disco")
        return joblib.load(ruta)
    t0 = time.perf_counter()
    resultado = funcion()
    if USAR_CACHE:
        joblib.dump(resultado, ruta, compress=3)
    print(f"[guardado] '{nombre}' calculado en {time.perf_counter() - t0:.0f} s")
    return resultado

def guardar_fig(nombre):
    plt.savefig(os.path.join(DIRS['images'], f'ev3_{nombre}.png'), dpi=120, bbox_inches='tight')

print(f"scikit-learn {sklearn.__version__} | xgboost {xgboost.__version__} | pandas {pd.__version__}")
print(f"MODO_RAPIDO = {MODO_RAPIDO} | Directorio raíz del proyecto: {os.path.abspath(ROOT)}")

In [ ]:
kpis = pd.read_csv(ubicar('kpis_spotify.csv'))

# Cómo se mide cada KPI en EV3 (trazabilidad KPI -> código)
medicion_ev3 = {
    'MAE':             'XGBRegressor sobre popularity, set de lanzamiento (Sección 10.3)',
    'RMSE':            'XGBRegressor sobre popularity, set de lanzamiento (Sección 10.3)',
    'R2':              'R² del regresor SOLO con variables musicales (Sección 10.3)',
    'Recall@Hit':      'Recall en test al umbral de la política (Sección 9)',
    'PR-AUC':          'Average Precision en CV y test (Secciones 7, 11)',
    'F2-Score':        'F-beta con beta=2 al umbral elegido',
    'NDCG@20':         'Calidad del ranking por género (Sección 13.4)',
    'Waste Rate':      'FP / (FP + TP) = 1 - Precision (restricción del umbral)',
    'Diversity Score': 'Nº de géneros en el top-200 promocionado (Sección 13.4)',
}
kpis['Medición en EV3'] = kpis['KPI'].map(medicion_ev3)
display(kpis[['KPI', 'Tipo de Analisis', 'Meta Sugerida', 'Medición en EV3']])

## 3. Carga, verificación, ingeniería de características y partición por artista

El código de esta sección es **idéntico al de EV2 corregida** (se genera desde la misma fuente), para que ambas
entregas trabajen con el mismo dataset, las mismas variables y el mismo conjunto de test:
- Validación defensiva, `is_hit`, género principal y las 7 variables derivadas de EV2.
- `artist_primary`: primer artista de la lista, usado como **grupo** en todas las particiones.
- `artist_n_tracks`: tamaño del catálogo del artista (metadato público, conocido al lanzar, sin información del target).
- `FEATURES_LANZAMIENTO`: audio + ingeniería + género (decidido con la ablación de EV2). `artist_n_tracks`,
  `artists` y `album_name` se conservan solo para la ablación y para auditar por tipo de artista.

In [ ]:
df = pd.read_csv(ubicar('spotify_tracks_clean.csv'))

core_audio = ['explicit', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
              'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']
n_antes = len(df)
df = df.dropna(subset=core_audio).reset_index(drop=True)
df['explicit']   = df['explicit'].astype(int)
df['is_hit']     = (df['popularity'] >= UMBRAL_HIT).astype(int)
df['artists']    = df['artists'].fillna('desconocido').astype(str)
df['album_name'] = df['album_name'].fillna('desconocido').astype(str)

genre_cols = [c for c in df.columns if c.startswith('genre_')]
activos = df[genre_cols].sum(axis=1)
df['genre'] = df[genre_cols].idxmax(axis=1).str.replace('genre_', '', regex=False)
df.loc[activos == 0, 'genre'] = 'unknown'

# Artista principal (primer nombre de la lista): define los GRUPOS del split.
# Se usa el principal y no el string completo para que "A" y "A;B" queden del mismo lado.
df['artist_primary'] = df['artists'].str.split(';').str[0].str.strip()

# Ingeniería de características (EV2)
df['duration_min']     = df['duration_ms'] / 60000
df['is_short']         = (df['duration_min'] < 3).astype(int)
df['is_long']          = (df['duration_min'] > 5).astype(int)
df['n_artists']        = df['artists'].str.split(';').str.len()
df['energy_x_dance']   = df['energy'] * df['danceability']
df['valence_x_energy'] = df['valence'] * df['energy']
df['explicit_x_dance'] = df['explicit'] * df['danceability']
# Tamaño del catálogo del artista principal (metadato público, sin información del target).
# Se usa para ANALIZAR el desempeño por tipo de artista; la ablación de EV2 mostró que como variable no aporta.
df['artist_n_tracks']  = df.groupby('artist_primary')['track_id'].transform('size')

audio_features = ['duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode',
                  'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence',
                  'tempo', 'time_signature']
engineered = ['duration_min', 'is_short', 'is_long', 'n_artists',
              'energy_x_dance', 'valence_x_energy', 'explicit_x_dance']
catalogo   = ['artist_n_tracks']
num_cols   = audio_features + engineered            # numéricas "musicales"
FEATURES_LANZAMIENTO = num_cols + genre_cols              # set final: disponible al lanzar (decidido en EV2, Sección 8)
cat_cols   = ['artists', 'album_name']              # solo para la ablación (Target Encoding)
X_cols     = FEATURES_LANZAMIENTO + catalogo + cat_cols   # catálogo y TE solo para ablación/auditoría

assert df['is_hit'].isin([0, 1]).all(), 'Target mal formado'
assert df[FEATURES_LANZAMIENTO].notna().all().all(), 'Nulos en variables del modelo'
print(f"Filas: {n_antes:,} -> {len(df):,} | Géneros one-hot: {len(genre_cols)} | "
      f"Artistas principales: {df['artist_primary'].nunique():,}")
print(f"Prevalencia de hit global: {df['is_hit'].mean():.4f}")
print(f"Canciones repetidas (mismo nombre y artista, distinto track_id): "
      f"{df.duplicated(['track_name', 'artists']).sum():,}")

### 3.1 Partición train/test por artista (idéntica a EV2)
`GroupShuffleSplit` con grupo = artista principal: **ningún artista aparece en train y test**. Esto elimina tres
fuentes de fuga detectadas en la retroalimentación de EV2: canciones hermanas del mismo artista, canciones del
mismo álbum y la misma canción publicada con otro `track_id`. El test queda aislado: no se usa para elegir
hiperparámetros, umbral ni modelo; solo para la confirmación final.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
idx_tr, idx_te = next(gss.split(df, df['is_hit'], groups=df['artist_primary']))
df_train, df_test = df.iloc[idx_tr].copy(), df.iloc[idx_te].copy()

X_train, X_test = df_train[X_cols].copy(), df_test[X_cols].copy()
y_train, y_test = df_train['is_hit'].copy(), df_test['is_hit'].copy()
g_train, g_test = df_train['genre'].copy(), df_test['genre'].copy()
grupos_train    = df_train['artist_primary'].to_numpy()

# Verificaciones de la partición
solapados = set(df_train['artist_primary']) & set(df_test['artist_primary'])
assert len(solapados) == 0, 'Hay artistas en train y test'
albums_compartidos = df_test['album_name'].isin(set(df_train['album_name'])).mean()
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"Train: {len(df_train):,} canciones / {df_train['artist_primary'].nunique():,} artistas")
print(f"Test : {len(df_test):,} canciones / {df_test['artist_primary'].nunique():,} artistas (ninguno visto en train)")
print(f"Hit rate train: {y_train.mean():.4f} | test: {y_test.mean():.4f}")
print(f"Géneros presentes en test: {g_test.nunique()} de {df['genre'].nunique()}")
print(f"Canciones de test cuyo nombre de álbum también aparece en train: {albums_compartidos:.1%} "
      "(compilados como 'Greatest Hits' o 'Mega Hits'; ver ablación)")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

## 4. Pipeline reproducible sin fuga de información

### Decisión: todo paso que "aprende" va dentro del `Pipeline`
La imputación, el escalamiento y (en la ablación) el `TargetEncoder` aprenden parámetros de los datos. Dentro de
un `Pipeline` se reajustan en cada pliegue de la validación cruzada usando solo las filas de entrenamiento de ese
pliegue; así la auditoría de las Secciones 11–12 no queda contaminada.

- `variante`: construye el mismo pipeline con distintos conjuntos de variables, usado en la ablación (10.3):
  `'musica'` (audio + ingeniería), `'lanzamiento'` (+ género, **modelo oficial**), `'catalogo'` (+ catálogo del
  artista, contraste) y `'con_te'` (+ Target Encoding de artista y álbum, **solo como contraste**: excluido por fuga).
- Todas las validaciones cruzadas usan `StratifiedGroupKFold` con grupo = artista.

In [ ]:
VARIANTES_COLS = {
    'musica':      num_cols,
    'lanzamiento': FEATURES_LANZAMIENTO,
    'catalogo':    FEATURES_LANZAMIENTO + catalogo,
    'con_te':      FEATURES_LANZAMIENTO + catalogo,
}

def construir_pipeline(modelo, variante='lanzamiento'):
    trans = [('num', 'passthrough', VARIANTES_COLS[variante])]
    if variante == 'con_te':
        trans.append(('te', TargetEncoder(smooth='auto', cv=5, random_state=RANDOM_STATE), cat_cols))
    return Pipeline([
        ('prep', ColumnTransformer(trans, remainder='drop')),
        ('imp',  SimpleImputer(strategy='median')),
        ('esc',  StandardScaler()),
        ('clf',  modelo),
    ])


def nombres_features(pipe):
    mapa = {'te__artists': 'artist_te', 'te__album_name': 'album_te'}
    return [mapa.get(n, n.split('__', 1)[1]) for n in pipe.named_steps['prep'].get_feature_names_out()]


class VotacionSuave(ClassifierMixin, BaseEstimator):
    # Ensamble de votación suave: promedia predict_proba de varios modelos.
    # Equivale a VotingClassifier(voting='soft'); además permite reutilizar modelos ya entrenados
    # (desde_ajustados) para no reentrenar RF y XGB una segunda vez.
    def __init__(self, estimadores=None, pesos=None):
        self.estimadores = estimadores
        self.pesos = pesos
    def fit(self, X, y):
        self.estimadores_ = [clone(e).fit(X, y) for e in self.estimadores]
        self.classes_ = np.unique(y)
        return self
    @classmethod
    def desde_ajustados(cls, estimadores, pesos=None):
        obj = cls(estimadores, pesos)
        obj.estimadores_ = list(estimadores)
        obj.classes_ = np.array([0, 1])
        return obj
    def predict_proba(self, X):
        w = np.ones(len(self.estimadores_)) if self.pesos is None else np.asarray(self.pesos, float)
        return sum(wi * e.predict_proba(X) for wi, e in zip(w, self.estimadores_)) / w.sum()
    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]

print(f"Pipeline listo. Variables del modelo oficial (set de lanzamiento): {len(FEATURES_LANZAMIENTO)}")

## 5. Optimización de hiperparámetros (IE9)

### Modelos base (configuración heredada de EV2)
- **Árbol de Decisión**: aprendiz individual de referencia. Sirve para medir cuánto aporta ensamblar (IE10).
- **Regresión Logística**: baseline lineal interpretable de EV2.
- **Random Forest base** y **XGBoost base**: exactamente los hiperparámetros fijados a mano en EV2.

### Estrategia de búsqueda y su justificación
| Decisión | Elección | Justificación |
|---|---|---|
| Método | `RandomizedSearchCV` | El espacio de XGB tiene 8 hiperparámetros continuos; una grilla completa (como el GridSearch de la ActividadML con un árbol) sería inviable. Con el mismo presupuesto, la búsqueda aleatoria explora más valores distintos de los parámetros que realmente importan (Bergstra & Bengio, 2012). |
| Métrica a optimizar | `average_precision` (PR-AUC) | No depende del umbral (el umbral se optimiza aparte en la Sección 9) y es la métrica correcta con clase minoritaria (23.7%). Optimizar F1 aquí mezclaría dos decisiones. |
| Datos | Submuestra estratificada del train (`SEARCH_FRAC`) | Reduce el costo ~2× sin cambiar el ranking de configuraciones; el modelo final se reentrena con el 100% del train. |
| Validación interna | `StratifiedGroupKFold(3)` por artista | Mismo criterio que la evaluación final (artistas nunca vistos); suficiente para ordenar candidatos; la auditoría rigurosa con K=5 se hace después sobre el modelo ya elegido. |
| Comparación justa | La configuración base se evalúa con **los mismos pliegues** | El Δ (optimizado − base) es atribuible solo a los hiperparámetros. |
| Control de sobreajuste | **Regla de 1 desviación estándar**: entre las configuraciones cuyo PR-AUC en CV está a menos de 1σ de la mejor, se elige la de **menor brecha train − CV** | Elegir solo por el máximo CV premia modelos que memorizan (en una primera ejecución, RF eligió `min_samples_leaf=1` con PR-AUC de train 0.998 y brecha 0.38). Las configuraciones dentro de 1σ son estadísticamente equivalentes, así que se prefiere la más simple y regularizada. |

In [ ]:
def modelo_arbol():
    return DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, class_weight='balanced',
                                  random_state=RANDOM_STATE)

def modelo_lr():
    return LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)

def modelo_rf(**kw):
    base = dict(n_estimators=N_ARBOLES_RF_BASE, max_depth=None, min_samples_leaf=5, max_features='sqrt',
                class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1)
    base.update(kw)
    return RandomForestClassifier(**base)

def modelo_xgb(**kw):
    base = dict(n_estimators=800, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                scale_pos_weight=scale_pos_weight, eval_metric='aucpr', tree_method='hist',
                random_state=RANDOM_STATE, n_jobs=-1)
    base.update(kw)
    return XGBClassifier(**base)

# Submuestra estratificada para la búsqueda y pliegues compartidos
X_busq, _, y_busq, _, gr_busq, _ = train_test_split(X_train, y_train, grupos_train, train_size=SEARCH_FRAC,
                                                    stratify=y_train, random_state=RANDOM_STATE)
cv_busq = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
print(f"Submuestra de búsqueda: {len(X_busq):,} filas | prevalencia {y_busq.mean():.4f}")

### 5.1 Random Forest — espacio de búsqueda
- `n_estimators`: más árboles reducen varianza con rendimientos decrecientes.
- `max_depth`, `min_samples_leaf`, `min_samples_split`: controlan la complejidad de cada árbol (sobreajuste).
- `max_features`: decorrelación entre árboles, la clave del bagging.

In [ ]:
espacio_rf = {
    'clf__n_estimators':      [100, 200, 300],
    'clf__max_depth':         [8, 12, 16, 24, None],
    'clf__min_samples_leaf':  [2, 5, 10, 20, 40],
    'clf__min_samples_split': [2, 10, 20],
    'clf__max_features':      ['sqrt', 'log2'],
}
def _buscar_rf():
    b = RandomizedSearchCV(construir_pipeline(modelo_rf()), espacio_rf, n_iter=N_ITER_RF,
                           scoring='average_precision', cv=cv_busq, random_state=RANDOM_STATE,
                           n_jobs=1, refit=False, return_train_score=True, verbose=1)
    b.fit(X_busq, y_busq, groups=gr_busq)
    base = cross_validate(construir_pipeline(modelo_rf()), X_busq, y_busq, cv=cv_busq,
                          scoring='average_precision', groups=gr_busq, return_train_score=True)
    return b, base
busq_rf, cv_rf_base = cacheado('busqueda_rf_v2', _buscar_rf)
print("Mejores hiperparámetros RF:", busq_rf.best_params_)

### 5.2 XGBoost — espacio de búsqueda
- `learning_rate` × `n_estimators`: tasas bajas con más árboles generalizan mejor (escala logarítmica).
- `max_depth`, `min_child_weight`, `gamma`: complejidad y poda de cada árbol.
- `subsample`, `colsample_bytree`: aleatoriedad estilo bagging dentro del boosting (reduce varianza).
- `reg_lambda`: regularización L2 de los pesos de las hojas.
- `scale_pos_weight` se mantiene fijo (decisión de balanceo de EV2).

In [ ]:
espacio_xgb = {
    'clf__n_estimators':     randint(200, 900),
    'clf__learning_rate':    loguniform(0.02, 0.15),
    'clf__max_depth':        randint(3, 8),
    'clf__min_child_weight': [1, 5, 10, 20, 40],
    'clf__subsample':        uniform(0.6, 0.4),
    'clf__colsample_bytree': uniform(0.4, 0.5),
    'clf__reg_lambda':       loguniform(1, 30),
    'clf__gamma':            [0, 1, 2, 5],
}
def _buscar_xgb():
    b = RandomizedSearchCV(construir_pipeline(modelo_xgb()), espacio_xgb, n_iter=N_ITER_XGB,
                           scoring='average_precision', cv=cv_busq, random_state=RANDOM_STATE,
                           n_jobs=1, refit=False, return_train_score=True, verbose=1)
    b.fit(X_busq, y_busq, groups=gr_busq)
    base = cross_validate(construir_pipeline(modelo_xgb()), X_busq, y_busq, cv=cv_busq,
                          scoring='average_precision', groups=gr_busq, return_train_score=True)
    return b, base
busq_xgb, cv_xgb_base = cacheado('busqueda_xgb_v2', _buscar_xgb)
print("Mejores hiperparámetros XGB:", busq_xgb.best_params_)

### 5.3 Resultado de la optimización: Δ base vs optimizado (mismos pliegues)
Se reporta la **mejor configuración por CV** y la **configuración elegida** con la regla de 1σ (menor brecha
train − CV entre las equivalentes). Como el modelo oficial ya no usa Target Encoding, la métrica de train que entrega
`RandomizedSearchCV` es honesta y puede usarse para medir la brecha.

In [ ]:
def elegir_config(busq):
    '''Regla de 1σ: entre las configuraciones equivalentes a la mejor, la de menor brecha train - CV.'''
    r = pd.DataFrame(busq.cv_results_)
    r['brecha'] = r['mean_train_score'] - r['mean_test_score']
    mejor = r.loc[r['mean_test_score'].idxmax()]
    candidatas = r[r['mean_test_score'] >= mejor['mean_test_score'] - mejor['std_test_score']]
    return mejor, candidatas.loc[candidatas['brecha'].idxmin()], len(candidatas)

def resumen_busqueda(busq, cv_base, nombre):
    mejor, elegida, n_cand = elegir_config(busq)
    base_cv, base_tr = cv_base['test_score'].mean(), cv_base['train_score'].mean()
    return {
        'Modelo': nombre,
        'PR-AUC CV base':           round(base_cv, 4),
        'Brecha base':              round(base_tr - base_cv, 4),
        'PR-AUC CV mejor config':   round(mejor['mean_test_score'], 4),
        'Brecha mejor config':      round(mejor['brecha'], 4),
        'PR-AUC CV elegida (1σ)':   round(elegida['mean_test_score'], 4),
        'Brecha elegida':           round(elegida['brecha'], 4),
        'Δ (opt - base)':           round(elegida['mean_test_score'] - base_cv, 4),
        'Δ brecha (opt - base)':    round(elegida['brecha'] - (base_tr - base_cv), 4),
        'σ CV elegida':             round(elegida['std_test_score'], 4),
        'Configs equivalentes (1σ)': n_cand,
        'Configuraciones probadas': len(busq.cv_results_['params']),
    }

tabla_ie9 = pd.DataFrame([resumen_busqueda(busq_rf, cv_rf_base, 'Random Forest'),
                          resumen_busqueda(busq_xgb, cv_xgb_base, 'XGBoost')]).set_index('Modelo')
display(tabla_ie9)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, (nombre, busq, base) in zip(axes, [('Random Forest', busq_rf, cv_rf_base),
                                           ('XGBoost', busq_xgb, cv_xgb_base)]):
    r = pd.DataFrame(busq.cv_results_).sort_values('mean_test_score').reset_index(drop=True)
    ax.errorbar(r.index, r['mean_test_score'], yerr=r['std_test_score'], fmt='o',
                color=COLORES[nombre], ms=6, capsize=3, label='Configuraciones probadas')
    ax.axhline(base['test_score'].mean(), color='black', ls='--', lw=1.5, label=f"Base EV2 ({base['test_score'].mean():.4f})")
    ax.set_title(f'{nombre}: PR-AUC en CV por configuración')
    ax.set_xlabel('Configuración (ordenadas de peor a mejor)'); ax.set_ylabel('PR-AUC (CV=3)')
    ax.legend(loc='lower right', fontsize=8)
plt.tight_layout(); guardar_fig('ie9_busqueda'); plt.show()

r_xgb = pd.DataFrame(busq_xgb.cv_results_)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for ax, p in zip(axes, ['param_clf__learning_rate', 'param_clf__max_depth', 'param_clf__n_estimators']):
    ax.scatter(r_xgb[p].astype(float), r_xgb['mean_test_score'], color=COLORES['XGBoost'], s=40)
    ax.set_xlabel(p.replace('param_clf__', '')); ax.grid(alpha=0.3)
axes[0].set_ylabel('PR-AUC (CV=3)')
plt.suptitle('XGBoost: sensibilidad del PR-AUC a cada hiperparámetro', fontweight='bold')
plt.tight_layout(); guardar_fig('ie9_sensibilidad_xgb'); plt.show()

# Gráfico: desempeño vs sobreajuste de cada configuración probada
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, (nombre, busq, base) in zip(axes, [('Random Forest', busq_rf, cv_rf_base), ('XGBoost', busq_xgb, cv_xgb_base)]):
    r = pd.DataFrame(busq.cv_results_); r['brecha'] = r['mean_train_score'] - r['mean_test_score']
    mejor, elegida, _ = elegir_config(busq)
    ax.scatter(r['brecha'], r['mean_test_score'], color=COLORES[nombre], s=45, alpha=0.7, label='Configuraciones')
    ax.scatter([mejor['brecha']], [mejor['mean_test_score']], s=160, facecolors='none', edgecolors='black', lw=2, label='Mejor CV')
    ax.scatter([elegida['brecha']], [elegida['mean_test_score']], s=160, marker='*', color='black', label='Elegida (1σ)')
    ax.scatter([base['train_score'].mean() - base['test_score'].mean()], [base['test_score'].mean()], marker='s', s=80,
               color='white', edgecolors='black', label='Base EV2')
    ax.axvline(0.10, color='#c0392b', ls=':', lw=1.2, label='Brecha 0.10')
    ax.set_xlabel('Brecha train − CV (sobreajuste)'); ax.set_ylabel('PR-AUC en CV')
    ax.set_title(f'{nombre}: desempeño vs sobreajuste'); ax.legend(fontsize=7.5)
plt.tight_layout(); guardar_fig('ie9_desempeno_vs_sobreajuste'); plt.show()

for nombre, fila in tabla_ie9.iterrows():
    print(f"{nombre}: Δ PR-AUC = {fila['Δ (opt - base)']:+.4f} | brecha {fila['Brecha base']:.3f} (base) -> "
          f"{fila['Brecha elegida']:.3f} (elegida) | mejor-CV tenía brecha {fila['Brecha mejor config']:.3f}")

**Lectura y decisión.** La optimización persigue dos objetivos: no perder PR-AUC frente a la configuración manual de
EV2 y **reducir el sobreajuste**. La configuración elegida se adopta si su PR-AUC en CV no es peor que la base más
allá del ruido (1σ); en ese caso la ganancia es de **generalización** (menor brecha), aunque el PR-AUC quede igual.
Si no cumple, se conserva la base; la siguiente celda lo aplica automáticamente. La sensibilidad de XGB muestra qué hiperparámetros
mueven realmente la métrica (típicamente `learning_rate` y `max_depth`), lo que justifica haber usado una búsqueda
continua en lugar de una grilla gruesa.

In [ ]:
def params_limpios(busq):
    return {k.replace('clf__', ''): v for k, v in elegir_config(busq)[1]['params'].items()}

def conviene(nombre):
    f = tabla_ie9.loc[nombre]
    return f['Δ (opt - base)'] >= -f['σ CV elegida']   # no peor que la base más allá del ruido

usa_opt_rf  = conviene('Random Forest')
usa_opt_xgb = conviene('XGBoost')
PARAMS_RF  = params_limpios(busq_rf)  if usa_opt_rf  else {}
PARAMS_XGB = params_limpios(busq_xgb) if usa_opt_xgb else {}
print("RF final  :", 'optimizado' if usa_opt_rf  else 'base EV2', PARAMS_RF)
print("XGB final :", 'optimizado' if usa_opt_xgb else 'base EV2', PARAMS_XGB)

## 6. Ensambles y entrenamiento de los modelos finales (IE10)

| Técnica | Modelo | Mecanismo | Qué problema ataca |
|---|---|---|---|
| — (referencia) | Árbol de Decisión | Un solo árbol | Línea base para medir el aporte del ensamble |
| **Bagging** | Random Forest | Árboles en paralelo sobre muestras bootstrap y subconjuntos de variables; promedia | **Varianza** (un árbol profundo memoriza ruido) |
| **Boosting** | XGBoost | Árboles en serie; cada uno corrige el error residual del anterior, con regularización L2 | **Sesgo** (patrones no lineales e interacciones finas) |
| **Votación suave** | RF + XGB | Promedio de probabilidades de ambos | Combina errores **poco correlacionados** de bagging y boosting |

La votación se construye con los RF y XGB ya entrenados (`VotacionSuave.desde_ajustados`), lo que es
matemáticamente idéntico a `VotingClassifier(voting='soft')` pero evita entrenarlos dos veces.
Se entrenan también las versiones base de RF y XGB para reportar el Δ de IE9 sobre el test.

In [ ]:
definiciones = {
    'Árbol de Decisión':       construir_pipeline(modelo_arbol()),
    'Regresión Logística':     construir_pipeline(modelo_lr()),
    'Random Forest (base)':    construir_pipeline(modelo_rf()),
    'XGBoost (base)':          construir_pipeline(modelo_xgb()),
    'Random Forest':           construir_pipeline(modelo_rf(**PARAMS_RF)),
    'XGBoost':                 construir_pipeline(modelo_xgb(**PARAMS_XGB)),
}
def entrenar_modelo(nombre):
    pipe = clone(definiciones[nombre])          # copia: 'definiciones' queda sin entrenar para las CV
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    return pipe, time.perf_counter() - t0, pipe.predict_proba(X_test)[:, 1]

def slug(nombre):
    import unicodedata, re
    s_ = unicodedata.normalize('NFKD', nombre).encode('ascii', 'ignore').decode().lower()
    return re.sub(r'[^a-z0-9]+', '_', s_).strip('_')

modelos, t_fit, proba_test = {}, {}, {}
for nombre in definiciones:
    if nombre == 'Random Forest (base)':
        # Solo se necesitan sus predicciones (Δ de IE9): se guardan sin el modelo, que pesa cientos de MB
        t_fit[nombre], proba_test[nombre] = cacheado('pred_rf_base', lambda: entrenar_modelo('Random Forest (base)')[1:])
    elif nombre == 'Random Forest':
        # El Random Forest final no se guarda en caché (archivo muy grande): se reentrena en cada ejecución
        modelos[nombre], t_fit[nombre], proba_test[nombre] = entrenar_modelo(nombre)
    else:
        modelos[nombre], t_fit[nombre], proba_test[nombre] = cacheado(f'modelo_{slug(nombre)}', lambda: entrenar_modelo(nombre))
    print(f"{nombre:24s} entrenamiento: {t_fit[nombre]:6.1f} s")

modelos['Votación RF+XGB'] = VotacionSuave.desde_ajustados([modelos['Random Forest'], modelos['XGBoost']])
t_fit['Votación RF+XGB'] = t_fit['Random Forest'] + t_fit['XGBoost']
proba_test['Votación RF+XGB'] = modelos['Votación RF+XGB'].predict_proba(X_test)[:, 1]

FINALES = ['Árbol de Decisión', 'Regresión Logística', 'Random Forest', 'XGBoost', 'Votación RF+XGB']
print("\nModelos finales comparados:", FINALES)

## 7. Comparación lado a lado (métricas, Δ, FN/FP)

Primero se comparan los modelos con el **umbral por defecto 0.50** (como en la ActividadML), junto con las métricas
que no dependen del umbral (PR-AUC, ROC-AUC, Brier). En la Sección 9 se repite la comparación con el umbral
óptimo de negocio.

- **FN (hits perdidos)**: costo de oportunidad — la discográfica deja pasar un éxito.
- **FP (promoción desperdiciada)**: costo presupuestario — se invierte en una canción que no será hit.
- **Waste Rate** = FP / (FP + TP): fracción del presupuesto de promoción que se pierde.

In [ ]:
def metricas(y_true, proba, umbral=0.5):
    yhat = (proba >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, yhat, labels=[0, 1]).ravel()
    return {
        'Umbral':     umbral,
        'Accuracy':   accuracy_score(y_true, yhat),
        'Precision':  precision_score(y_true, yhat, zero_division=0),
        'Recall':     recall_score(y_true, yhat, zero_division=0),
        'F1':         f1_score(y_true, yhat, zero_division=0),
        'F2':         fbeta_score(y_true, yhat, beta=2, zero_division=0),
        'Waste Rate': fp / (fp + tp) if (fp + tp) else 0.0,
        'PR-AUC':     average_precision_score(y_true, proba),
        'ROC-AUC':    roc_auc_score(y_true, proba),
        'Brier':      brier_score_loss(y_true, proba),
        'TP': tp, 'FP (promoción perdida)': fp, 'FN (hits perdidos)': fn, 'TN': tn,
    }

comp_05 = pd.DataFrame({n: metricas(y_test, proba_test[n], 0.5) for n in proba_test}).T
print(f"Prevalencia en test (PR-AUC de un clasificador aleatorio): {y_test.mean():.4f}\n")
print("=== COMPARACIÓN EN TEST (umbral 0.50) ===")
display(comp_05.loc[FINALES].drop(columns='Umbral').round(4))

cols_delta = ['Precision', 'Recall', 'F1', 'F2', 'PR-AUC', 'ROC-AUC', 'FP (promoción perdida)', 'FN (hits perdidos)']
delta_arbol = comp_05.loc[FINALES, cols_delta].astype(float).sub(
    comp_05.loc['Árbol de Decisión', cols_delta].astype(float), axis=1)
print("=== Δ vs Árbol de Decisión (aporte del ensamble; positivo = mejor, salvo FP/FN) ===")
display(delta_arbol.round(4))

delta_ie9 = pd.DataFrame({
    'Random Forest': comp_05.loc['Random Forest', cols_delta].astype(float) - comp_05.loc['Random Forest (base)', cols_delta].astype(float),
    'XGBoost':       comp_05.loc['XGBoost', cols_delta].astype(float) - comp_05.loc['XGBoost (base)', cols_delta].astype(float),
}).T
print("=== Δ en test: optimizado - base EV2 (IE9) ===")
display(delta_ie9.round(4))

In [ ]:
met_graf = ['Precision', 'Recall', 'F1', 'PR-AUC', 'ROC-AUC']
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), gridspec_kw={'width_ratios': [2.2, 1]})
x = np.arange(len(met_graf)); w = 0.16
for i, n in enumerate(FINALES):
    axes[0].bar(x + (i - 2) * w, comp_05.loc[n, met_graf].astype(float), width=w * 0.92,
                color=COLORES[n], label=n)
axes[0].set_xticks(x); axes[0].set_xticklabels(met_graf)
axes[0].set_ylim(0, 1.05); axes[0].set_ylabel('Valor en test')
axes[0].set_title('Métricas en test (umbral 0.50)')
axes[0].legend(fontsize=8, ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.08))

err = comp_05.loc[FINALES, ['FN (hits perdidos)', 'FP (promoción perdida)']].astype(int)
xi = np.arange(len(FINALES))
axes[1].bar(xi - 0.2, err['FN (hits perdidos)'], 0.38, color='#c0392b', label='FN: hits perdidos')
axes[1].bar(xi + 0.2, err['FP (promoción perdida)'], 0.38, color='#f39c12', label='FP: promoción perdida')
axes[1].set_xticks(xi); axes[1].set_xticklabels([n.replace(' ', '\n', 1) for n in FINALES], fontsize=8)
axes[1].set_ylabel('Nº de canciones'); axes[1].set_title('Errores absolutos (umbral 0.50)')
axes[1].legend(fontsize=8)
plt.tight_layout(); guardar_fig('comparacion_umbral05'); plt.show()

### 7.1 Matrices de confusión comparadas (umbral 0.50)

In [ ]:
def matrices(umbrales, titulo, archivo):
    fig, axes = plt.subplots(1, len(FINALES), figsize=(4 * len(FINALES), 3.8))
    for ax, n in zip(axes, FINALES):
        u = umbrales[n] if isinstance(umbrales, dict) else umbrales
        cm = confusion_matrix(y_test, (proba_test[n] >= u).astype(int), labels=[0, 1])
        ConfusionMatrixDisplay(cm, display_labels=['No hit', 'Hit']).plot(ax=ax, cmap='Blues', colorbar=False,
                                                                         values_format=',d')
        tn, fp, fn, tp = cm.ravel()
        ax.set_title(f"{n} (u={u:.2f})\nFN={fn:,}  FP={fp:,}\nRecall={tp/(tp+fn):.3f}  Waste={fp/max(fp+tp,1):.3f}",
                     fontsize=9)
        ax.grid(False)
    plt.suptitle(titulo, fontweight='bold'); plt.tight_layout(); guardar_fig(archivo); plt.show()

matrices(0.5, 'Matrices de confusión en test — umbral 0.50', 'matrices_umbral05')

**Lectura.** Con 0.50 los modelos con balanceo de clases (`class_weight`, `scale_pos_weight`) ya empujan las
probabilidades hacia la clase hit, por lo que el recall suele ser alto, pero a costa de un **Waste Rate muy superior
al 15%** permitido por el negocio (EV2 reportó 39%). El umbral 0.50 no es una decisión de negocio, es un valor por
defecto: por eso la Sección 9 lo reemplaza por un umbral elegido según los KPIs.

## 8. Motor de validación cruzada K=5 (probabilidades fuera de pliegue)

Una sola pasada de `StratifiedGroupKFold(5)` **agrupada por artista** sobre el train alimenta tres secciones, evitando
reentrenar (en cada pliegue, los artistas de validación nunca se vieron en entrenamiento):
1. **Sección 9** – el umbral óptimo se elige con las probabilidades **OOF** (fuera de pliegue), nunca con el test.
2. **Sección 11** – métricas por pliegue (media, σ, estabilidad).
3. **Sección 12** – métricas en train de cada pliegue para el diagnóstico Train vs CV.

Para la votación suave se promedian las probabilidades OOF de RF y XGB de **los mismos pliegues**, lo que equivale
a entrenar el ensamble dentro de cada pliegue.

In [ ]:
skf = StratifiedGroupKFold(n_splits=K_FOLDS, shuffle=True, random_state=RANDOM_STATE)
PLIEGUES = list(skf.split(X_train, y_train, grupos_train))
y_tr_np = y_train.to_numpy()

def ajustar_honesto(pipe, X_a, y_a, X_b):
    '''Equivale a pipe.fit(X_a, y_a) + predict_proba(X_b), pero devuelve además la probabilidad en
    TRAIN calculada sobre la representación que el modelo realmente vio (Target Encoding cruzado),
    sin la auto-fuga que ocurre al llamar transform() sobre las mismas filas de ajuste.'''
    prep, clf = clone(pipe[:-1]), clone(pipe[-1])
    Xt_a = prep.fit_transform(X_a, y_a)
    clf.fit(Xt_a, y_a)
    return clf.predict_proba(Xt_a)[:, 1], clf.predict_proba(prep.transform(X_b))[:, 1]

def ejecutar_cv(pipe, nombre):
    oof = np.zeros(len(y_train))
    p_train, t_folds = [], []
    for k, (tr, va) in enumerate(PLIEGUES):
        t0 = time.perf_counter()
        p_tr, p_va = ajustar_honesto(pipe, X_train.iloc[tr], y_train.iloc[tr], X_train.iloc[va])
        t_folds.append(time.perf_counter() - t0)
        oof[va] = p_va
        p_train.append(p_tr)
    print(f"{nombre:22s} CV K={K_FOLDS} completada en {sum(t_folds):6.1f} s | "
          f"PR-AUC OOF = {average_precision_score(y_train, oof):.4f}")
    return {'oof': oof, 'p_train': p_train, 't_fold': np.mean(t_folds)}

CV = {}
for n in ['Árbol de Decisión', 'Regresión Logística', 'Random Forest', 'XGBoost']:
    CV[n] = cacheado(f'cv_k5_{slug(n)}', lambda: ejecutar_cv(definiciones[n], n))

CV['Votación RF+XGB'] = {
    'oof':     (CV['Random Forest']['oof'] + CV['XGBoost']['oof']) / 2,
    'p_train': [(a + b) / 2 for a, b in zip(CV['Random Forest']['p_train'], CV['XGBoost']['p_train'])],
    't_fold':  CV['Random Forest']['t_fold'] + CV['XGBoost']['t_fold'],
}
print(f"{'Votación RF+XGB':22s} PR-AUC OOF = {average_precision_score(y_train, CV['Votación RF+XGB']['oof']):.4f}")

## 9. Curvas ROC y Precision-Recall comparadas

- **ROC**: capacidad global de discriminación. Con desbalance puede verse optimista, porque la tasa de falsos
  positivos se diluye entre los muchos negativos.
- **Precision-Recall**: la curva relevante para el negocio; su eje Y es exactamente `1 − Waste Rate`. La línea base
  es la prevalencia (≈0.24). La banda sombreada marca la zona permitida por el KPI (Precision ≥ 0.85).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for n in FINALES:
    fpr, tpr, _ = roc_curve(y_test, proba_test[n])
    axes[0].plot(fpr, tpr, lw=2, color=COLORES[n], label=f"{n} (AUC={roc_auc_score(y_test, proba_test[n]):.3f})")
    prec, rec, _ = precision_recall_curve(y_test, proba_test[n])
    axes[1].plot(rec, prec, lw=2, color=COLORES[n],
                 label=f"{n} (AP={average_precision_score(y_test, proba_test[n]):.3f})")
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Azar')
axes[0].set_xlabel('Tasa de falsos positivos'); axes[0].set_ylabel('Recall (TPR)')
axes[0].set_title('Curva ROC (test)'); axes[0].legend(fontsize=8, loc='lower right')

axes[1].axhspan(1 - WASTE_MAX, 1.0, color='#1baf7a', alpha=0.08, label=f'Zona KPI (Waste ≤ {WASTE_MAX:.0%})')
axes[1].axvline(RECALL_MIN, color='black', ls=':', lw=1.2, label=f'Recall mínimo KPI ({RECALL_MIN:.0%})')
axes[1].axhline(y_test.mean(), color='k', ls='--', alpha=0.4, label=f'Azar (prevalencia={y_test.mean():.2f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision = 1 − Waste Rate')
axes[1].set_title('Curva Precision-Recall (test, clase Hit)'); axes[1].legend(fontsize=7.5, loc='lower left')
axes[1].set_ylim(0, 1.02)
plt.tight_layout(); guardar_fig('curvas_roc_pr'); plt.show()

**Lectura.** Si las curvas PR cruzan la intersección entre la zona sombreada y la línea vertical de recall 70%,
existe un umbral que cumple **ambos** KPIs a la vez. Si ninguna curva llega a esa esquina, los KPIs de
Recall y Waste Rate están en conflicto y el negocio debe priorizar uno de ellos: esa es exactamente la decisión que
formaliza la siguiente sección.

## 9.1 Umbral óptimo con código (maximizar Recall bajo restricción)

**Política de negocio adoptada:** *garantizar Recall ≥ 70% (KPI Recall@Hit) con el menor Waste Rate posible* —
"descubrir al menos 7 de cada 10 hits gastando lo mínimo". Es la misma regla de la ActividadML (recall mínimo y,
entre los umbrales que lo cumplen, el de menos falsos positivos).

**Por qué no "Waste ≤ 15%":** esa meta se fijó cuando el modelo tenía fuga de información. Con artistas nunca
vistos e información disponible al lanzar, un Waste ≤ 15% solo se logra promocionando una fracción mínima de canciones
(Recall muy bajo): la política quedaría inútil. La tabla de políticas lo muestra y la Sección 9.1 calcula el
*trade-off* exacto. La referencia correcta para el desperdicio es el **azar**: promocionar al azar desperdicia
1 − prevalencia ≈ 76% del presupuesto.

Se comparan cuatro políticas para cada modelo, todas evaluadas sobre las probabilidades **OOF del train**:

| Política | Regla |
|---|---|
| Por defecto | umbral 0.50 |
| Máx F2 (EV2) | umbral que maximiza F2 |
| Máx Recall s.a. Waste ≤ 15% | entre los umbrales con Waste ≤ 0.15, el de mayor Recall (meta original, se muestra su costo) |
| **Mín Waste s.a. Recall ≥ 70%** | entre los umbrales con Recall ≥ 0.70, el de menor Waste (**política oficial**) |

> **Corrección de sesgo respecto a la ActividadML:** allí el barrido de umbral se hizo sobre `y_test`, por lo que
> el desempeño reportado incluía la ventaja de haber "mirado" el test. Aquí el umbral se fija con OOF y el test
> se usa una sola vez para confirmarlo.

In [ ]:
GRILLA_U = np.round(np.arange(0.05, 0.951, 0.01), 2)

def barrido_umbral(y_true, proba, umbrales=GRILLA_U):
    y_true = np.asarray(y_true)
    filas = []
    for u in umbrales:
        yhat = (proba >= u).astype(int)
        tp = int(((yhat == 1) & (y_true == 1)).sum()); fp = int(((yhat == 1) & (y_true == 0)).sum())
        fn = int(((yhat == 0) & (y_true == 1)).sum())
        prec = tp / (tp + fp) if (tp + fp) else 0.0; rec = tp / (tp + fn) if (tp + fn) else 0.0
        f2 = 5 * prec * rec / (4 * prec + rec) if (prec + rec) else 0.0
        filas.append({'umbral': u, 'Precision': prec, 'Recall': rec, 'F2': f2,
                      'Waste Rate': (fp / (tp + fp)) if (tp + fp) else 0.0, 'FP': fp, 'FN': fn})
    return pd.DataFrame(filas)

def politicas(d):
    out = {'Por defecto (0.50)': (0.50, True),
           'Máx F2 (EV2)': (d.loc[d['F2'].idxmax(), 'umbral'], True)}
    ok = d[d['Waste Rate'] <= WASTE_MAX]
    out['Máx Recall s.a. Waste ≤ 15%'] = ((ok.sort_values(['Recall', 'umbral'], ascending=[False, True]).iloc[0]['umbral'], True)
                                          if len(ok) else (d.loc[d['Waste Rate'].idxmin(), 'umbral'], False))
    ok2 = d[d['Recall'] >= RECALL_MIN]
    out['Mín Waste s.a. Recall ≥ 70%'] = ((ok2.sort_values(['Waste Rate', 'umbral']).iloc[0]['umbral'], True)
                                          if len(ok2) else (d.loc[d['Recall'].idxmax(), 'umbral'], False))
    return out

POLITICA = 'Mín Waste s.a. Recall ≥ 70%'
barridos, tabla_pol, UMBRAL = {}, [], {}
for n in FINALES:
    d = barrido_umbral(y_train, CV[n]['oof']); barridos[n] = d
    for pol, (u, factible) in politicas(d).items():
        mt = metricas(y_test, proba_test[n], u)
        tabla_pol.append({'Modelo': n, 'Política': pol, 'Umbral': u, 'Factible en OOF': factible,
                          'Recall OOF': d.loc[d['umbral'] == u, 'Recall'].iloc[0] if u in set(d['umbral']) else np.nan,
                          'Waste OOF': d.loc[d['umbral'] == u, 'Waste Rate'].iloc[0] if u in set(d['umbral']) else np.nan,
                          'Recall test': mt['Recall'], 'Waste test': mt['Waste Rate'], 'F2 test': mt['F2'],
                          'FN test': mt['FN (hits perdidos)'], 'FP test': mt['FP (promoción perdida)']})
        if pol == POLITICA:
            UMBRAL[n] = float(u)

tabla_pol = pd.DataFrame(tabla_pol)
display(tabla_pol.set_index(['Modelo', 'Política']).round(3))
print("Umbral oficial por modelo (", POLITICA, "):", {k: round(v, 2) for k, v in UMBRAL.items()})

In [ ]:
fig, axes = plt.subplots(1, len(FINALES), figsize=(4.2 * len(FINALES), 4), sharey=True)
for ax, n in zip(axes, FINALES):
    d = barridos[n]
    ax.plot(d['umbral'], d['Recall'], lw=2, color=COLORES[n], label='Recall')
    ax.plot(d['umbral'], d['Precision'], lw=2, color='black', ls='--', label='Precision')
    ax.plot(d['umbral'], d['Waste Rate'], lw=1.5, color='#c0392b', label='Waste Rate')
    ax.fill_between(d['umbral'], 0, 1, where=d['Waste Rate'] <= WASTE_MAX, color='#1baf7a', alpha=0.08,
                    label='Waste ≤ 15%')
    ax.axhline(RECALL_MIN, color='gray', ls=':', lw=1)
    ax.axvline(UMBRAL[n], color=COLORES[n], lw=1.5)
    ax.axvline(0.5, color='gray', ls='--', lw=0.8, alpha=0.6)
    ax.set_title(f"{n}\numbral óptimo = {UMBRAL[n]:.2f}", fontsize=10); ax.set_xlabel('Umbral')
axes[0].set_ylabel('Métrica (OOF train)'); axes[0].legend(fontsize=7, loc='center left')
plt.suptitle('Barrido de umbral sobre probabilidades fuera de pliegue (línea vertical gris = 0.50)', fontweight='bold')
plt.tight_layout(); guardar_fig('barrido_umbral'); plt.show()

# ¿Los KPIs de Recall y Waste son compatibles?
print("=== COMPATIBILIDAD DE KPIs (OOF) ===")
for n in FINALES:
    d = barridos[n]
    ambos = d[(d['Recall'] >= RECALL_MIN) & (d['Waste Rate'] <= WASTE_MAX)]
    if len(ambos):
        print(f"{n:22s} ✅ existen {len(ambos)} umbrales que cumplen Recall ≥ 70% y Waste ≤ 15% "
              f"({ambos['umbral'].min():.2f}–{ambos['umbral'].max():.2f})")
    else:
        r_max = d.loc[d['Waste Rate'] <= WASTE_MAX, 'Recall'].max()
        w_min = d.loc[d['Recall'] >= RECALL_MIN, 'Waste Rate'].min()
        print(f"{n:22s} ⚠️ conflicto: con Waste ≤ 15% el Recall máximo es {r_max if pd.notna(r_max) else 0:.3f}; "
              f"para lograr Recall ≥ 70% el Waste mínimo sería {w_min:.3f}")

### 9.2 Comparación final al umbral óptimo de negocio (test)
Todos los modelos operan ahora con el **mismo nivel de descubrimiento** (Recall ≈ 70% en OOF). La pregunta de
negocio se vuelve directa: *para descubrir 7 de cada 10 hits, ¿qué modelo desperdicia menos presupuesto?*
Referencia: promocionar al azar desperdicia 1 − prevalencia del presupuesto.

In [ ]:
comp_opt = pd.DataFrame({n: metricas(y_test, proba_test[n], UMBRAL[n]) for n in FINALES}).T
display(comp_opt.round(4))

# Comparación independiente del umbral: desperdicio necesario para descubrir el 70% de los hits (curva PR del test)
def waste_a_recall(y, p, r=RECALL_MIN):
    prec, rec, _ = precision_recall_curve(y, p)
    return 1 - prec[rec >= r].max()
WASTE70 = {n: waste_a_recall(y_test, proba_test[n]) for n in FINALES}
print(f"Waste Rate necesario para Recall {RECALL_MIN:.0%} (leído en la curva PR del test, sin depender del umbral):")
for n in FINALES:
    aviso = ("  ⚠️ su umbral OOF no se transfirió al test (Recall test "
             f"{comp_opt.loc[n, 'Recall']:.2f}): probabilidades escalonadas e inestables"
             if comp_opt.loc[n, 'Recall'] < RECALL_MIN - 0.10 else "")
    print(f"  {n:22s} {WASTE70[n]:.3f}{aviso}")
print(f"  {'Azar':22s} {1 - y_test.mean():.3f}")

delta_opt = comp_opt[cols_delta].astype(float).sub(comp_opt.loc['Árbol de Decisión', cols_delta].astype(float), axis=1)
print("=== Δ vs Árbol de Decisión al umbral óptimo (aporte del ensamble) ===")
display(delta_opt.round(4))

d_umbral = (comp_opt[['Recall', 'Waste Rate', 'FN (hits perdidos)', 'FP (promoción perdida)']].astype(float)
            - comp_05.loc[FINALES, ['Recall', 'Waste Rate', 'FN (hits perdidos)', 'FP (promoción perdida)']].astype(float))
print("=== Efecto del umbral: óptimo - 0.50 ===")
display(d_umbral.round(4))

print(f"Waste Rate si se promocionara al azar: {1 - y_test.mean():.3f}")
matrices(UMBRAL, 'Matrices de confusión en test — umbral óptimo (Recall ≥ 70% con mínimo Waste)', 'matrices_umbral_opt')

**Lectura.** El umbral de negocio fija el Recall en ≈70% (cumple el KPI Recall@Hit). La tabla "óptimo − 0.50"
muestra cuánto cambian FP y FN respecto del umbral por defecto. Como todos los modelos descubren la misma proporción
de hits, la diferencia entre ellos está en **cuánto presupuesto desperdician para lograrlo**; ahí se ve el aporte real
de los ensambles frente al árbol simple y la regresión logística, y la distancia a la referencia del azar.

## 10. Feature Importance con código (RF + XGB + permutation)

Tres métodos complementarios, porque cada uno tiene un sesgo conocido:
- **Gini (RF)**: rápido, pero **favorece variables continuas y de alta cardinalidad** (p. ej. `artist_te`).
- **Gain (XGB)**: ganancia media de las divisiones; puede concentrarse en pocas variables muy informativas.
- **Permutación (test)**: caída del PR-AUC al desordenar una variable; es agnóstico al modelo y se mide en datos no
  vistos, por eso es el árbitro. Se calcula sobre las variables del set de lanzamiento.

In [ ]:
feat_names = nombres_features(modelos['Random Forest'])
imp_rf = (pd.DataFrame({'feature': feat_names,
                        'importance': modelos['Random Forest'].named_steps['clf'].feature_importances_})
          .sort_values('importance', ascending=False).reset_index(drop=True))
imp_xgb = (pd.DataFrame({'feature': nombres_features(modelos['XGBoost']),
                         'importance': modelos['XGBoost'].named_steps['clf'].feature_importances_})
           .sort_values('importance', ascending=False).reset_index(drop=True))

# Permutación sobre una muestra estratificada del test (costo controlado)
X_perm, _, y_perm, _ = train_test_split(X_test, y_test, train_size=min(6000, len(X_test) - 1),
                                        stratify=y_test, random_state=RANDOM_STATE)
X_perm = X_perm[FEATURES_LANZAMIENTO]
t0 = time.perf_counter()
perm = cacheado('permutacion_xgb', lambda: permutation_importance(
    modelos['XGBoost'], X_perm, y_perm, scoring='average_precision',
    n_repeats=3 if MODO_RAPIDO else 5, random_state=RANDOM_STATE, n_jobs=1))
imp_perm = (pd.DataFrame({'feature': X_perm.columns, 'importance': perm.importances_mean,
                          'std': perm.importances_std})
            .sort_values('importance', ascending=False).reset_index(drop=True))
print(f"Permutation importance calculada en {time.perf_counter() - t0:.0f} s")

for titulo, d in [('Random Forest — Gini', imp_rf), ('XGBoost — Gain', imp_xgb),
                  ('Permutación (XGBoost, caída de PR-AUC en test)', imp_perm)]:
    print(f"\n=== TOP 10 — {titulo} ===")
    print(d.head(10).round(4).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
for ax, (titulo, d, color) in zip(axes, [('Random Forest — Gini', imp_rf, COLORES['Random Forest']),
                                         ('XGBoost — Gain', imp_xgb, COLORES['XGBoost']),
                                         ('Permutación (XGBoost)', imp_perm, '#555555')]):
    top = d.head(10).iloc[::-1]
    ax.barh(top['feature'], top['importance'], color=color, height=0.6)
    if 'std' in top:
        ax.errorbar(top['importance'], top['feature'], xerr=top['std'], fmt='none', ecolor='black', capsize=3)
    ax.set_title(titulo); ax.set_xlabel('Importancia')
plt.tight_layout(); guardar_fig('feature_importance'); plt.show()

top3 = {'RF': set(imp_rf.head(3)['feature']), 'XGB': set(imp_xgb.head(3)['feature']),
        'Perm': set(imp_perm.head(3)['feature'].replace({'artists': 'artist_te', 'album_name': 'album_te'}))}
for k, v in top3.items():
    print(f"Top-3 {k:5s}: {sorted(v)}")
print("Intersección de los tres métodos:", top3['RF'] & top3['XGB'] & top3['Perm'] or '— (sin consenso total)')

### 10.1 Importancia por bloques de variables (¿música o género?)
Las 114 columnas de género se reparten la importancia entre sí y por separado parecen irrelevantes; agrupadas
pueden pesar mucho. Por eso se permutan **bloques completos** (todas las columnas del bloque con la misma
permutación de filas). Esto cuantifica el **sesgo de retroalimentación / efecto Mateo** descrito en EV1:
si el bloque "Género" domina sobre "Acústicas", el modelo predice sobre todo *en qué mercado* compite la canción,
no *cómo suena*.

In [ ]:
BLOQUES = {
    'Acústicas (14)':            audio_features,
    'Ingeniería (7)':            engineered,
    f'Género ({len(genre_cols)} one-hot)': genre_cols,
}

def importancia_bloques(modelo, X, y, n_rep=5, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    base = average_precision_score(y, modelo.predict_proba(X)[:, 1])
    filas = []
    for bloque, cols in BLOQUES.items():
        caidas = []
        for _ in range(n_rep):
            Xp = X.copy()
            idx = rng.permutation(len(Xp))
            Xp[cols] = Xp[cols].iloc[idx].to_numpy()
            caidas.append(base - average_precision_score(y, modelo.predict_proba(Xp)[:, 1]))
        filas.append({'Bloque': bloque, 'Caída PR-AUC': np.mean(caidas), 'std': np.std(caidas)})
    return pd.DataFrame(filas), base

imp_bloques = {}
for n in ['Random Forest', 'XGBoost', 'Votación RF+XGB']:
    d, base = cacheado(f'bloques_{slug(n)}', lambda: importancia_bloques(modelos[n], X_perm, y_perm,
                                                                         n_rep=3 if MODO_RAPIDO else 5))
    d['Modelo'] = n; d['% de la caída total'] = 100 * d['Caída PR-AUC'] / d['Caída PR-AUC'].clip(lower=0).sum()
    imp_bloques[n] = d
tabla_bloques = pd.concat(imp_bloques.values()).set_index(['Modelo', 'Bloque'])
display(tabla_bloques.round(4))

fig, ax = plt.subplots(figsize=(10, 4))
bl = list(BLOQUES); x = np.arange(len(bl)); w = 0.27
for i, n in enumerate(imp_bloques):
    d = imp_bloques[n].set_index('Bloque').loc[bl]
    ax.bar(x + (i - 1) * w, d['Caída PR-AUC'], w * 0.92, yerr=d['std'], capsize=3, color=COLORES[n], label=n)
ax.set_xticks(x); ax.set_xticklabels(bl); ax.set_ylabel('Caída de PR-AUC al permutar')
ax.set_title('Importancia por bloque de variables (test)'); ax.legend(fontsize=8)
plt.tight_layout(); guardar_fig('importancia_bloques'); plt.show()

### 10.2 Traducción del top de variables a acción de negocio

In [ ]:
acciones = {
    'artist_te':    'Historial del artista: es señal de exposición previa (marketing, seguidores). Usarlo para '
                    'priorizar, pero con cuota protegida para artistas emergentes (sesgo de retroalimentación).',
    'artists':      'Ídem artist_te: la identidad del artista domina la predicción.',
    'album_te':     'Desempeño del álbum: canciones de álbumes con tracción heredan visibilidad (efecto arrastre).',
    'album_name':   'Ídem album_te.',
    'artist_count': 'Volumen de catálogo del artista: artistas prolíficos tienen más exposición acumulada.',
    'artist_n_tracks': 'Catálogo del artista: los artistas establecidos concentran hits (exposición acumulada). '
                       'Priorizar con cuidado: refuerza el efecto Mateo; reservar cuota para artistas nuevos.',
    'duration_ms':  'Duración: privilegiar cortes de radio (≈3–4 min) para singles promocionales.',
    'duration_min': 'Duración: privilegiar cortes de radio (≈3–4 min) para singles promocionales.',
    'danceability': 'Bailabilidad: candidata a playlists de fiesta/fitness.',
    'energy':       'Energía: alinear con playlists de alta rotación.',
    'loudness':     'Masterización: revisar estándar de loudness de la plataforma.',
    'acousticness': 'Perfil acústico: suele asociarse a catálogo de nicho; evaluar con métricas por género.',
    'instrumentalness': 'Instrumental: menor tasa de hit; usar playlists temáticas (estudio, relajación).',
    'speechiness':  'Contenido hablado: separar podcasts/comedia del análisis musical.',
    'valence':      'Positividad: perfil clásico de hit pop.',
    'n_artists':    'Colaboraciones: evaluar featurings con artistas de mayor base de seguidores.',
}
def accion(feat):
    if feat.startswith('genre_'):
        return (f"Género '{feat[6:]}': su peso refleja la prevalencia de hits del género en la plataforma "
                "(sesgo de mercado), no calidad musical. Monitorear con métricas por género.")
    return acciones.get(feat, 'Revisar con el equipo de A&R.')

for titulo, d in [('Random Forest', imp_rf), ('XGBoost', imp_xgb), ('Permutación', imp_perm)]:
    print(f"\n===== REPORTE DE NEGOCIO — {titulo} =====")
    for i, row in d.head(3).iterrows():
        print(f"{i + 1}. {row['feature']:18s} (imp={row['importance']:.4f})\n   → {accion(row['feature'])}")

### 10.3 Ablación: ¿cuánto del éxito se debe solo a la música? (KPI R² / MAE / RMSE)
El KPI **R²** de `kpis_spotify.csv` se define como *"porcentaje de la popularidad que se debe solo a la música y no
al marketing"*. Se entrenan cuatro variantes del mismo pipeline (mismos hiperparámetros de XGB), evaluadas en el test
de **artistas nunca vistos**:

1. **Solo música**: 14 variables de audio + 7 derivadas.
2. **Lanzamiento**: agrega el one-hot de género (contexto de mercado) → **modelo oficial**.
3. **+ Catálogo del artista**: contraste (excluido en EV2 porque no aporta con artistas nuevos y premia el tamaño del catálogo).
4. **+ TE artista y álbum**: contraste para mostrar la fuga detectada en la retroalimentación de EV2 (**excluido**).

Se mide en clasificación (PR-AUC) y en regresión sobre `popularity` (MAE, RMSE, R²) con `XGBRegressor`.

In [ ]:
# Regresor con los MISMOS hiperparámetros que EV2 (Sección 17) para que los KPIs MAE/RMSE/R² coincidan entre notebooks
params_reg = dict(n_estimators=600, learning_rate=0.05, max_depth=7, subsample=0.8, colsample_bytree=0.8)
pop_train, pop_test = df_train['popularity'], df_test['popularity']
ETQ = {'musica': '1. Solo música', 'lanzamiento': '2. Lanzamiento (+ género)',
       'catalogo': '3. + Catálogo (excluido)', 'con_te': '4. + TE artista/álbum (excluido)'}

def _variante(variante):
    clf_v = construir_pipeline(modelo_xgb(**PARAMS_XGB), variante).fit(X_train, y_train)
    reg_v = construir_pipeline(XGBRegressor(tree_method='hist', random_state=RANDOM_STATE, n_jobs=-1, **params_reg),
                               variante).fit(X_train, pop_train)
    return clf_v.predict_proba(X_test)[:, 1], np.clip(reg_v.predict(X_test), 0, 100)

filas_abl, modelos_reg = [], {}
for variante, etiqueta in ETQ.items():
    t0 = time.perf_counter()
    p_v, pred_pop = cacheado(f'ablacion_{variante}', lambda: _variante(variante))
    modelos_reg[etiqueta] = (None, pred_pop)
    filas_abl.append({'Variante': etiqueta, 'PR-AUC (clasif.)': average_precision_score(y_test, p_v),
                      'ROC-AUC (clasif.)': roc_auc_score(y_test, p_v),
                      'MAE (popularidad)': mean_absolute_error(pop_test, pred_pop),
                      'RMSE (popularidad)': np.sqrt(mean_squared_error(pop_test, pred_pop)),
                      'R² (popularidad)': r2_score(pop_test, pred_pop),
                      'seg': round(time.perf_counter() - t0, 1)})
ablacion = pd.DataFrame(filas_abl).set_index('Variante')
display(ablacion.round(4))

colores_abl = ['#9ec5f4', '#1c5cab', '#bbbbbb', '#bbbbbb']
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
etq_corta = [t.replace(' (', '\n(') for t in ablacion.index]
axes[0].bar(etq_corta, ablacion['PR-AUC (clasif.)'], color=colores_abl, width=0.55)
axes[0].axhline(y_test.mean(), color='k', ls='--', lw=1, label='Azar'); axes[0].set_ylim(0, 1)
axes[0].set_title('PR-AUC por variante (test, artistas nuevos)'); axes[0].legend()
axes[1].bar(etq_corta, ablacion['R² (popularidad)'], color=colores_abl, width=0.55)
axes[1].axhline(0.45, color='#c0392b', ls=':', lw=1.5, label='Meta KPI R² (45%)'); axes[1].set_ylim(0, 1)
axes[1].set_title('R² sobre popularidad por variante'); axes[1].legend()
for ax in axes:
    ax.tick_params(axis='x', labelsize=7.5)
plt.tight_layout(); guardar_fig('ablacion'); plt.show()

R2_MUSICA = ablacion.loc[ETQ['musica'], 'R² (popularidad)']
R2_LANZ   = ablacion.loc[ETQ['lanzamiento'], 'R² (popularidad)']
print(f"La música por sí sola explica R² = {R2_MUSICA:.1%} de la popularidad; con el género, {R2_LANZ:.1%}.")
print(f"El TE de artista/álbum cambia el PR-AUC en "
      f"{ablacion.loc[ETQ['con_te'], 'PR-AUC (clasif.)'] - ablacion.loc[ETQ['lanzamiento'], 'PR-AUC (clasif.)']:+.3f} "
      "aun con artistas nunca vistos: es fuga por álbumes compartidos (compilados), no capacidad predictiva.")

**Lectura.** La diferencia entre "solo música" y "lanzamiento" es la evidencia cuantitativa del **sesgo de
retroalimentación** (EV1): gran parte de la capacidad predictiva viene del **contexto de mercado** (qué géneros premia
la plataforma), más que de la calidad sonora. Consecuencias: (1) el modelo sirve para
**priorizar**, pero (2) no debe usarse como filtro para artistas nuevos sin una política de equidad (Sección 13).
La variante 4 confirma la observación del docente: el Target Encoding de álbum "mejora" las métricas porque los
álbumes compartidos (compilados, playlists y reediciones como "Mega Hits" o "Christmas Songs 2022") tienen popularidad
muy homogénea: la media del álbum adivina la etiqueta de la canción, y solo se conoce después del lanzamiento.

## 11. Auditoría estadística con validación cruzada K=5

Criterio de estabilidad (igual que la ActividadML): **σ ≤ 0.05** entre pliegues indica que el rendimiento no
depende de la partición y que la generalización es certificable. Se audita el PR-AUC (independiente del umbral)
y el Recall / Waste al umbral oficial de cada modelo.

*Nota:* el Recall/Waste por pliegue usa el umbral elegido con todas las OOF, por lo que es levemente optimista;
el test (Sección 9.2) es la confirmación independiente.

In [ ]:
def por_pliegue(n):
    filas = []
    for k, (tr, va) in enumerate(PLIEGUES):
        yv, pv = y_tr_np[va], CV[n]['oof'][va]
        m = metricas(yv, pv, UMBRAL[n])
        filas.append({'fold': k + 1, 'PR-AUC': m['PR-AUC'], 'ROC-AUC': m['ROC-AUC'], 'Recall': m['Recall'],
                      'Waste Rate': m['Waste Rate'], 'F2': m['F2']})
    return pd.DataFrame(filas)

audit, folds_df = [], {}
for n in FINALES:
    f = por_pliegue(n); folds_df[n] = f
    fila = {'Modelo': n}
    for k in range(K_FOLDS):
        fila[f'PR-AUC_f{k + 1}'] = f.loc[k, 'PR-AUC']
    for met in ['PR-AUC', 'ROC-AUC', 'Recall', 'Waste Rate', 'F2']:
        fila[f'{met}_media'] = f[met].mean(); fila[f'{met}_std'] = f[met].std(ddof=0)
    fila['Estabilidad'] = ('✅ ESTABLE' if max(fila['PR-AUC_std'], fila['Recall_std']) <= 0.05 else '⚠️ INESTABLE')
    audit.append(fila)
cv_audit = pd.DataFrame(audit).set_index('Modelo')
print(f"=== CV K={K_FOLDS} (StratifiedGroupKFold por artista, train) ===")
display(cv_audit.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for n in FINALES:
    f = folds_df[n]
    axes[0].plot(f['fold'], f['PR-AUC'], marker='o', ms=8, lw=2, color=COLORES[n],
                 label=f"{n} (μ={f['PR-AUC'].mean():.3f}, σ={f['PR-AUC'].std(ddof=0):.3f})")
    axes[1].plot(f['fold'], f['Recall'], marker='s', ms=8, lw=2, color=COLORES[n],
                 label=f"{n} (μ={f['Recall'].mean():.3f}, σ={f['Recall'].std(ddof=0):.3f})")
axes[0].axhline(y_train.mean(), color='gray', ls='--', lw=1, label='Azar')
axes[1].axhline(RECALL_MIN, color='gray', ls=':', lw=1, label='Meta Recall 70%')
axes[0].set_title('PR-AUC por pliegue'); axes[1].set_title('Recall por pliegue (umbral oficial)')
for ax in axes:
    ax.set_xlabel('Pliegue'); ax.set_xticks(range(1, K_FOLDS + 1)); ax.legend(fontsize=7.5)
plt.tight_layout(); guardar_fig('cv_k5'); plt.show()

def interpretar_varianza(std, metrica, umbral=0.05):
    if std <= umbral:
        return f"σ({metrica})={std:.4f} ≤ {umbral}: ESTABLE, el rendimiento es consistente entre particiones."
    return (f"σ({metrica})={std:.4f} > {umbral}: ALTA VARIANZA, el rendimiento depende de la muestra; "
            "riesgo de comportamiento errático con canciones nuevas.")

print("=== DIAGNÓSTICO DE ESTABILIDAD ===")
for n, r in cv_audit.iterrows():
    print(f"{n:22s} {interpretar_varianza(r['PR-AUC_std'], 'PR-AUC')}")
    print(f"{'':22s} {interpretar_varianza(r['Recall_std'], 'Recall')}")

### 11.1 Intervalo de confianza de la diferencia entre los dos mejores modelos
Una diferencia de medias pequeña puede ser ruido. Se calcula la diferencia de PR-AUC **pliegue a pliegue**
(los modelos comparten pliegues, es una comparación pareada) y su intervalo aproximado al 95% (t de Student, 4 g.l.).
Si el intervalo contiene el 0, los modelos están en **empate estadístico** y la elección debe apoyarse en otros
criterios (estabilidad, equidad, costo, interpretabilidad).

In [ ]:
from scipy import stats
orden = cv_audit['PR-AUC_media'].sort_values(ascending=False).index.tolist()
m1, m2 = orden[0], orden[1]
dif = folds_df[m1]['PR-AUC'].to_numpy() - folds_df[m2]['PR-AUC'].to_numpy()
t_crit = stats.t.ppf(0.975, df=K_FOLDS - 1)
ic = (dif.mean() - t_crit * dif.std(ddof=1) / np.sqrt(K_FOLDS), dif.mean() + t_crit * dif.std(ddof=1) / np.sqrt(K_FOLDS))
print(f"Mejor en CV: {m1} | Segundo: {m2}")
print(f"Diferencia media de PR-AUC = {dif.mean():+.4f} | IC95% = [{ic[0]:+.4f}, {ic[1]:+.4f}]")
print("=> Diferencia estadísticamente distinta de 0" if ic[0] > 0 or ic[1] < 0
      else "=> Empate estadístico: la elección se decide con los demás criterios de la matriz de decisión")

### 11.2 ¿Cuánto inflaba una validación cruzada aleatoria?
Toda la auditoría usa `StratifiedGroupKFold` (artistas de validación nunca vistos). Para cuantificar la corrección
pedida en la retroalimentación de EV2, se repite la CV del modelo con una `StratifiedKFold` **aleatoria**, que
mezcla canciones del mismo artista (y repeticiones de la misma canción) entre entrenamiento y validación.
La diferencia es el **optimismo** que tendría el reporte si no se agrupara por artista.

In [ ]:
skf_aleatorio = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=RANDOM_STATE)
modelos_fuga = ['XGBoost']   # se usa XGBoost (rápido) como representante; RF mostraría el mismo patrón
def _oof_aleatorio(n):
    oof = np.zeros(len(y_train))
    for tr, va in skf_aleatorio.split(X_train, y_train):
        m = clone(definiciones[n]).fit(X_train.iloc[tr], y_train.iloc[tr])
        oof[va] = m.predict_proba(X_train.iloc[va])[:, 1]
    return oof
fuga, oof_aleatorio = [], {}
for n in modelos_fuga:
    t0 = time.perf_counter()
    oof = cacheado(f'cv_aleatoria_{slug(n)}', lambda: _oof_aleatorio(n))
    oof_aleatorio[n] = oof
    ma, mg = metricas(y_train, oof, UMBRAL[n]), metricas(y_train, CV[n]['oof'], UMBRAL[n])
    fuga.append({'Modelo': n, 'PR-AUC CV por artista': mg['PR-AUC'], 'PR-AUC CV aleatoria': ma['PR-AUC'],
                 'Inflación PR-AUC': ma['PR-AUC'] - mg['PR-AUC'], 'Recall por artista': mg['Recall'],
                 'Recall aleatoria': ma['Recall'], 'seg': round(time.perf_counter() - t0)})
tabla_fuga = pd.DataFrame(fuga).set_index('Modelo')
display(tabla_fuga.round(4))
print("Lectura: la inflación es lo que la versión anterior reportaba de más. Aun sin Target Encoding existe "
      "algo de inflación, porque el modelo reconoce el estilo y las canciones repetidas de artistas ya vistos.")

## 12. Diagnóstico de sobreajuste (Train vs. CV)

Bitácora de control técnico (criterio ActividadML): **brecha = métrica_train − métrica_CV**.
- |brecha| < 0.10 → ✅ OK · brecha ≥ 0.10 → ⚠️ OVERFITTING · brecha ≤ −0.10 → ⚠️ UNDERFITTING.

### Hallazgo de la verificación: auto-fuga del Target Encoding en train
En la primera versión de EV3 (que aún usaba Target Encoding de artista y álbum), la métrica de train calculada con
`pipeline.predict_proba(X_train)` usaba para cada canción la media de su artista **incluyendo su propia etiqueta**:
incluso la Regresión Logística, que no puede memorizar, mostraba PR-AUC de train ≈ 0.99. Fue una segunda razón para
retirar el TE del modelo oficial. Como salvaguarda, el train se sigue midiendo con `ajustar_honesto` (Sección 8):
sobre la representación que el modelo realmente vio al entrenar.

*Acción aplicada contra el sobreajuste:* desde la Sección 5, la búsqueda de hiperparámetros elige, entre las
configuraciones estadísticamente equivalentes en CV, la de **menor brecha train − CV** (regla de 1σ), con espacios que
incluyen regularización fuerte (`max_depth`, `min_samples_leaf` en RF; `min_child_weight`, `gamma`, `reg_lambda` en XGB).

*Matiz adicional:* un Random Forest con hojas pequeñas casi memoriza el train **por diseño** (PR-AUC_train ≈ 1);
lo que valida a un RF no es su train sino que su CV sea alto y estable, porque el promedio de árboles cancela la
varianza individual. Por eso el diagnóstico se interpreta junto con la σ de la Sección 11.

In [ ]:
bitacora = []
for n in FINALES:
    tr_prauc = np.mean([average_precision_score(y_tr_np[tr], p) for (tr, _), p in zip(PLIEGUES, CV[n]['p_train'])])
    tr_f2 = np.mean([fbeta_score(y_tr_np[tr], (p >= UMBRAL[n]).astype(int), beta=2)
                     for (tr, _), p in zip(PLIEGUES, CV[n]['p_train'])])
    for met, tr_val, cv_val, cv_std in [('PR-AUC', tr_prauc, cv_audit.loc[n, 'PR-AUC_media'], cv_audit.loc[n, 'PR-AUC_std']),
                                        ('F2', tr_f2, cv_audit.loc[n, 'F2_media'], cv_audit.loc[n, 'F2_std'])]:
        br = tr_val - cv_val
        bitacora.append({'Modelo': n, 'Métrica': met, 'Train': tr_val, 'CV_media': cv_val, 'CV_std': cv_std,
                         'Brecha': br,
                         'Diagnóstico': '✅ OK' if abs(br) < 0.10 else ('⚠️ OVERFITTING' if br > 0 else '⚠️ UNDERFITTING')})
bitacora_df = pd.DataFrame(bitacora).set_index(['Modelo', 'Métrica'])
print("=== BITÁCORA DE CONTROL TÉCNICO ===")
display(bitacora_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, met in zip(axes, ['PR-AUC', 'F2']):
    sub = bitacora_df.xs(met, level='Métrica').loc[FINALES]
    x = np.arange(len(sub))
    ax.bar(x - 0.2, sub['Train'], 0.38, color='#86b6ef', label='Train')
    ax.bar(x + 0.2, sub['CV_media'], 0.38, color='#1c5cab', label='CV media', yerr=sub['CV_std'], capsize=4)
    for xi, b in zip(x, sub['Brecha']):
        ax.text(xi, 1.02, f"Δ={b:+.2f}", ha='center', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels([n.replace(' ', '\n', 1) for n in sub.index], fontsize=8)
    ax.set_ylim(0, 1.1); ax.set_title(f'{met}: Train vs CV'); ax.legend(loc='lower right', fontsize=8)
plt.tight_layout(); guardar_fig('train_vs_cv'); plt.show()

# Nota: es esperable que test sea levemente MAYOR que CV, porque el modelo final se entrena con el 100% del
# train y en CV cada modelo usa solo el 80%. Se tolera |Δ| < 0.03.
print("=== CONSISTENCIA CV vs TEST (segunda prueba de generalización) ===")
for n in FINALES:
    d_ = comp_opt.loc[n, 'PR-AUC'] - cv_audit.loc[n, 'PR-AUC_media']
    print(f"{n:22s} PR-AUC CV={cv_audit.loc[n, 'PR-AUC_media']:.4f} | test={comp_opt.loc[n, 'PR-AUC']:.4f} | Δ={d_:+.4f} -> "
          + ("✅ el desempeño en datos nunca vistos coincide con la CV" if abs(d_) < 0.03 else "⚠️ discrepancia, revisar"))

### 12.1 Curvas de aprendizaje
- Si la curva de train se mantiene alta y la de CV se estanca lejos → **sobreajuste** (memoriza).
- Si ambas convergen a un valor bajo → **subajuste** (modelo simple).
- Si la brecha se cierra al aumentar datos → más datos mejorarían el modelo; la arquitectura es correcta.

In [ ]:
cv3 = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
tamanos = np.linspace(0.15, 1.0, 4 if MODO_RAPIDO else 5)
modelos_lc = ['Árbol de Decisión', 'XGBoost']   # aprendiz simple vs ensamble (RF se omite por costo)
fig, axes = plt.subplots(1, len(modelos_lc), figsize=(6 * len(modelos_lc), 4.2), sharey=True)
def curva_aprendizaje(pipe, fracs):
    '''learning_curve manual con medición de train honesta (ver Sección 12).'''
    sz, tr_s, cv_s = [], [], []
    for f in fracs:
        tr_f, cv_f = [], []
        for tr, va in cv3.split(X_train, y_train, grupos_train):
            sub = tr if f >= 1 else train_test_split(tr, train_size=f, stratify=y_tr_np[tr],
                                                     random_state=RANDOM_STATE)[0]
            p_tr, p_va = ajustar_honesto(pipe, X_train.iloc[sub], y_train.iloc[sub], X_train.iloc[va])
            tr_f.append(average_precision_score(y_tr_np[sub], p_tr))
            cv_f.append(average_precision_score(y_tr_np[va], p_va))
        sz.append(len(sub)); tr_s.append(tr_f); cv_s.append(cv_f)
    return np.array(sz), np.array(tr_s), np.array(cv_s)

curvas = {}
for ax, n in zip(np.atleast_1d(axes), modelos_lc):
    t0 = time.perf_counter()
    sz, tr_s, cv_s = cacheado(f'curva_{slug(n)}', lambda: curva_aprendizaje(definiciones[n], tamanos))
    curvas[n] = (sz, tr_s.mean(1), cv_s.mean(1))
    for s, lab, ls in [(tr_s, 'Train', '--'), (cv_s, 'CV', '-')]:
        ax.plot(sz, s.mean(1), ls, marker='o', ms=7, lw=2, color=COLORES[n], label=lab)
        ax.fill_between(sz, s.mean(1) - s.std(1), s.mean(1) + s.std(1), color=COLORES[n], alpha=0.12)
    ax.set_title(f"{n} ({time.perf_counter() - t0:.0f} s)"); ax.set_xlabel('Canciones de entrenamiento')
    ax.legend(fontsize=8)
np.atleast_1d(axes)[0].set_ylabel('PR-AUC')
plt.suptitle('Curvas de aprendizaje (CV=3)', fontweight='bold'); plt.tight_layout(); guardar_fig('curvas_aprendizaje'); plt.show()

for n, (sz, tr_m, cv_m) in curvas.items():
    brecha_ini, brecha_fin = tr_m[0] - cv_m[0], tr_m[-1] - cv_m[-1]
    tend = 'se cierra' if brecha_fin < brecha_ini - 0.01 else 'se mantiene'
    print(f"{n:20s} brecha inicial {brecha_ini:.3f} -> final {brecha_fin:.3f} ({tend}); "
          f"ganancia de CV en el último tramo: {cv_m[-1] - cv_m[-2]:+.4f}")

**Acciones de ingeniería si se detecta sobreajuste** (aplicadas o disponibles en este proyecto):
1. **Regularización**: `min_samples_leaf`, `max_depth` (RF); `reg_lambda`, `gamma`, `min_child_weight` (XGB) — ya
   incluidos en el espacio de búsqueda de la Sección 5, que eligió con CV y no con train.
2. **Ensamble (bagging/votación)**: promediar modelos reduce la varianza; la votación RF+XGB es la aplicación directa.
3. **Submuestreo de filas y columnas** en XGB (`subsample`, `colsample_bytree`).
4. **Suavizado del Target Encoding** (`smooth='auto'`, `cv=5`): evita que artistas con una sola canción se
   codifiquen con su propia etiqueta (fuga a nivel de fila).
5. **Más datos / early stopping** como línea futura.

## 13. Análisis de sesgos y equidad con evidencia cuantitativa

EV1 identificó cinco sesgos (muestreo por duplicación, recencia, retroalimentación/efecto Mateo, homogeneización
cultural, prevalencia desigual por género). EV2 cuantificó la prevalencia. EV3 mide **cómo se comporta la solución
operativa (al umbral de negocio)** frente a esos sesgos.

### 13.1 Equidad por grupo de géneros (mainstream vs nicho)
Los géneros se agrupan en terciles según su tasa de hit **en train**. Se compara el Recall al umbral oficial
con las probabilidades OOF (decisión) y en test (confirmación). Agrupar evita la inestabilidad del F1 por género
individual con 0–3 positivos documentada en EV2.

In [ ]:
tasa_genero = df_train.groupby('genre')['is_hit'].mean()
grupo_genero = pd.qcut(tasa_genero.rank(method='first'), 3, labels=['Nicho', 'Intermedio', 'Mainstream'])
grp_train = g_train.map(grupo_genero).astype(str).to_numpy()
grp_test  = g_test.map(grupo_genero).fillna('Nicho').astype(str).to_numpy()
print("Tasa de hit media por grupo (train):",
      df_train.assign(g=grp_train).groupby('g')['is_hit'].mean().round(3).to_dict())

def recall_grupo(y, p, u, grp, g):
    m = (grp == g) & (np.asarray(y) == 1)
    return ((p[m] >= u).mean()) if m.sum() else np.nan

equidad = []
for n in FINALES:
    fila = {'Modelo': n}
    for g in ['Nicho', 'Intermedio', 'Mainstream']:
        fila[f'Recall {g} (OOF)'] = recall_grupo(y_train, CV[n]['oof'], UMBRAL[n], grp_train, g)
        fila[f'Recall {g} (test)'] = recall_grupo(y_test, proba_test[n], UMBRAL[n], grp_test, g)
    fila['Ratio equidad Nicho/Mainstream (OOF)'] = fila['Recall Nicho (OOF)'] / fila['Recall Mainstream (OOF)']
    fila['Ratio equidad Nicho/Mainstream (test)'] = fila['Recall Nicho (test)'] / fila['Recall Mainstream (test)']
    equidad.append(fila)
tabla_equidad = pd.DataFrame(equidad).set_index('Modelo')
display(tabla_equidad.round(3))

fig, ax = plt.subplots(figsize=(11, 4))
gs_ = ['Nicho', 'Intermedio', 'Mainstream']; x = np.arange(3); w = 0.16
for i, n in enumerate(FINALES):
    ax.bar(x + (i - 2) * w, [tabla_equidad.loc[n, f'Recall {g} (test)'] for g in gs_], w * 0.92,
           color=COLORES[n], label=n)
ax.axhline(RECALL_MIN, color='gray', ls=':', lw=1, label='Meta Recall 70%')
ax.set_xticks(x); ax.set_xticklabels(gs_); ax.set_ylim(0, 1.05); ax.set_ylabel('Recall (test)')
ax.set_title('Recall por grupo de géneros al umbral de negocio'); ax.legend(fontsize=8, ncol=3)
plt.tight_layout(); guardar_fig('equidad_generos'); plt.show()

**Lectura.** Un ratio Nicho/Mainstream < 1 significa que, ante dos hits reales, el modelo tiene más probabilidad de
descubrir el de un género mainstream. Es la manifestación operativa de la **homogeneización cultural**: el umbral
único favorece a los géneros donde la plataforma ya concentra éxitos. Este ratio entra como criterio de **Equidad**
en la matriz de decisión (Sección 14), de modo que la selección del modelo no sea solo predictiva.

### 13.2 Artistas emergentes vs establecidos (tamaño del catálogo)
Todos los artistas del test son nuevos para el modelo, pero difieren en su catálogo publicado: 1 canción
(emergente o esporádico), 2–10 o más de 10 (establecido). Se compara el Recall al umbral oficial y la calibración
(probabilidad media predicha vs tasa real de hit).

In [ ]:
seg_art = pd.cut(df_test['artist_n_tracks'], bins=[0, 1, 10, np.inf],
                 labels=['1 canción', '2–10 canciones', '>10 canciones']).astype(str).to_numpy()
yt_np = y_test.to_numpy()
filas = []
for n in FINALES:
    for sg in ['1 canción', '2–10 canciones', '>10 canciones']:
        m = seg_art == sg; pos = m & (yt_np == 1); pred_pos = m & (proba_test[n] >= UMBRAL[n])
        filas.append({'Modelo': n, 'Catálogo': sg, 'Canciones': int(m.sum()), 'Tasa hit real': yt_np[m].mean(),
                      'Prob. media predicha': proba_test[n][m].mean(),
                      'Recall': (proba_test[n][pos] >= UMBRAL[n]).mean() if pos.sum() else np.nan,
                      'Waste Rate': (yt_np[pred_pos] == 0).mean() if pred_pos.sum() else np.nan})
tabla_artistas = pd.DataFrame(filas).set_index(['Modelo', 'Catálogo'])
display(tabla_artistas.round(3))

fig, ax = plt.subplots(figsize=(11, 4))
cats_a = ['1 canción', '2–10 canciones', '>10 canciones']; x = np.arange(3); w = 0.16
for i, n in enumerate(FINALES):
    ax.bar(x + (i - 2) * w, [tabla_artistas.loc[(n, c), 'Recall'] for c in cats_a], w * 0.92, color=COLORES[n], label=n)
ax.set_xticks(x); ax.set_xticklabels(cats_a); ax.set_ylim(0, 1); ax.set_ylabel('Recall (test)')
ax.set_title('Recall por tamaño de catálogo del artista (umbral de negocio)'); ax.legend(fontsize=8, ncol=3)
plt.tight_layout(); guardar_fig('artistas_emergentes'); plt.show()

### 13.3 Equidad por segmento sonoro (K-Means de EV2)
Se reutiliza el aprendizaje no supervisado de EV2 (PCA 85% de varianza + K-Means, K=4) sobre las variables
acústicas del train, para verificar si el modelo descarta sistemáticamente algún perfil sonoro.

In [ ]:
esc_audio = StandardScaler().fit(X_train[audio_features])
pca = PCA(n_components=0.85, random_state=RANDOM_STATE).fit(esc_audio.transform(X_train[audio_features]))
kmeans = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=20).fit(
    pca.transform(esc_audio.transform(X_train[audio_features])))
cl_test = kmeans.predict(pca.transform(esc_audio.transform(X_test[audio_features])))

filas = []
for c in range(4):
    m = cl_test == c
    fila = {'Cluster': c, 'n_test': int(m.sum()), 'Tasa hit real': y_test.to_numpy()[m].mean(),
            'energy': X_test['energy'].to_numpy()[m].mean(), 'acousticness': X_test['acousticness'].to_numpy()[m].mean()}
    for n in FINALES:
        pos = m & (y_test.to_numpy() == 1)
        fila[f'Recall {n}'] = (proba_test[n][pos] >= UMBRAL[n]).mean() if pos.sum() else np.nan
    filas.append(fila)
tabla_cluster = pd.DataFrame(filas).set_index('Cluster')
display(tabla_cluster.round(3))

### 13.4 Calidad del ranking (NDCG@20) y diversidad de la lista promocionada
- **NDCG@20**: para cada género (una "playlist candidata") se ordenan las canciones por probabilidad y se compara
  con el orden real de popularidad. 1.0 = orden perfecto en el top-20.
- **Diversity Score**: número de géneros distintos en el **top-200** de canciones que el modelo promocionaría
  (presupuesto limitado) y entre todas las canciones clasificadas como hit.
- **Mitigación**: si el top-200 se concentra en pocos géneros, se aplica un **re-ranking con cupo por género**
  (máximo 10% de la lista por género) y se mide cuánta precisión cuesta ganar diversidad.

In [ ]:
def ndcg_por_genero(proba, k=20, min_n=40):
    vals = []
    for g in np.unique(g_test):
        m = (g_test.to_numpy() == g)
        if m.sum() >= min_n:
            vals.append(ndcg_score(pop_test.to_numpy()[m].reshape(1, -1), proba[m].reshape(1, -1), k=k))
    return np.mean(vals)

rank = []
for n in FINALES:
    top200 = np.argsort(-proba_test[n])[:200]
    rank.append({'Modelo': n, 'NDCG@20 (media por género)': ndcg_por_genero(proba_test[n]),
                 'Géneros en top-200': len(np.unique(g_test.to_numpy()[top200])),
                 'Géneros entre hits predichos': len(np.unique(g_test.to_numpy()[proba_test[n] >= UMBRAL[n]])),
                 '% top-200 del género más frecuente': pd.Series(g_test.to_numpy()[top200]).value_counts(normalize=True).iloc[0]})
tabla_rank = pd.DataFrame(rank).set_index('Modelo')
display(tabla_rank.round(3))

# --- Mitigación aplicada: re-ranking con cupo máximo por género ---
def rerank_diverso(proba, k=200, cupo=0.10):
    max_g = int(np.ceil(k * cupo)); cuenta, elegidos = {}, []
    for i in np.argsort(-proba):
        g = g_test.iloc[i]
        if cuenta.get(g, 0) < max_g:
            elegidos.append(i); cuenta[g] = cuenta.get(g, 0) + 1
        if len(elegidos) == k:
            break
    return np.array(elegidos)

yt_ = y_test.to_numpy()
filas_rr = []
for n in FINALES:
    top = np.argsort(-proba_test[n])[:200]; div = rerank_diverso(proba_test[n])
    filas_rr.append({'Modelo': n, 'Precisión top-200 original': yt_[top].mean(), 'Géneros original': g_test.iloc[top].nunique(),
                     'Precisión top-200 con cupo 10%': yt_[div].mean(), 'Géneros con cupo': g_test.iloc[div].nunique()})
tabla_rerank = pd.DataFrame(filas_rr).set_index('Modelo')
print("Re-ranking con cupo máximo de 10% por género en la lista de 200 canciones a promocionar:")
display(tabla_rerank.round(3))

### 13.5 Mapa de sesgos: evidencia y mitigación

In [ ]:
ref = 'XGBoost'
bl = imp_bloques[ref].set_index('Bloque')['% de la caída total']
ta = tabla_artistas.loc[ref]
sesgos = pd.DataFrame([
    {'Sesgo': 'Fuga de información en la evaluación (artistas/álbumes compartidos, TE de álbum, umbral en test)',
     'Evidencia en EV3': f"CV aleatoria infla el PR-AUC en {tabla_fuga.loc[ref, 'Inflación PR-AUC']:+.3f}; TE álbum "
                         f"{ablacion.loc[ETQ['con_te'], 'PR-AUC (clasif.)'] - ablacion.loc[ETQ['lanzamiento'], 'PR-AUC (clasif.)']:+.3f} con artistas nuevos",
     'Mitigación': 'Split y CV por artista; sin TE; umbral con OOF; test usado una vez'},
    {'Sesgo': 'Retroalimentación / efecto Mateo (popularidad = exposición previa)',
     'Evidencia en EV3': f"R² solo música = {R2_MUSICA:.2f} vs {R2_LANZ:.2f} con género; bloque Género = "
                         f"{bl.filter(like='Género').sum():.0f}% de la importancia",
     'Mitigación': 'Usar como señal complementaria; cuota de presupuesto para artistas sin historial'},
    {'Sesgo': 'Artistas emergentes vs establecidos',
     'Evidencia en EV3': f"Recall 1 canción = {ta.loc['1 canción', 'Recall']:.2f} vs >10 canciones = {ta.loc['>10 canciones', 'Recall']:.2f}"
                         + (" (no se subestima a emergentes)" if ta.loc['1 canción', 'Recall'] >= ta.loc['>10 canciones', 'Recall'] - 0.03 else " (se subestima a emergentes)"),
     'Mitigación': 'Sin TE ni catálogo como variables; monitoreo del recall por segmento de artista'},
    {'Sesgo': 'Homogeneización cultural / prevalencia por género',
     'Evidencia en EV3': f"Ratio Recall Nicho/Mainstream (test) = {tabla_equidad.loc[ref, 'Ratio equidad Nicho/Mainstream (test)']:.2f}",
     'Mitigación': 'Monitorear ratio por iteración; umbrales por grupo o cuotas de diversidad'},
    {'Sesgo': 'Concentración de la lista promocionada',
     'Evidencia en EV3': f"{tabla_rank.loc[ref, 'Géneros en top-200']} géneros en top-200; género dominante = {tabla_rank.loc[ref, '% top-200 del género más frecuente']:.0%}",
     'Mitigación': 'Re-ranking con restricción de diversidad (máx. % por género)'},
    {'Sesgo': 'Recencia y muestreo (≈1.000 canciones por género vía API; popularidad = reproducciones recientes)',
     'Evidencia en EV3': 'No medible con el dataset (no hay fecha de lanzamiento)',
     'Mitigación': 'Incorporar fecha de lanzamiento y re-entrenamiento periódico'},
]).set_index('Sesgo')
display(sesgos)

## 14. Sustentación de la solución óptima (IE12)

### 14.1 Matriz de decisión
**Regla metodológica:** todos los criterios cuantitativos provienen de la **validación cruzada** (no del test), para
no seleccionar el modelo con los mismos datos con los que se reporta su desempeño (sesgo de selección).

| Criterio | Fuente | Peso | Justificación de negocio |
|---|---|---|---|
| Precisión al umbral de negocio (1 − Waste) | CV K=5 | **0.30** | Con el Recall fijado en 70%, gana quien desperdicia menos presupuesto |
| PR-AUC | CV K=5 | **0.25** | Calidad del ranking independiente del umbral (robustez ante cambios de presupuesto) |
| Equidad (ratio Nicho/Mainstream) | OOF | **0.15** | Compromiso ético: no amplificar la homogeneización cultural |
| Estabilidad (1 − σ PR-AUC) | CV K=5 | **0.10** | Generalización certificable |
| Generalización (−brecha Train-CV) | CV K=5 | **0.10** | Penaliza sobreajuste |
| Interpretabilidad (1–5) | Cualitativo | **0.05** | Auditabilidad ante artistas y regulación (Ley 19.628, AI Act) |
| Costo de infraestructura (1–5, menor es mejor) | Cualitativo + tiempo | **0.05** | Operación y reentrenamiento |

In [ ]:
cualitativo = {  # (Interpretabilidad 1-5, Costo infraestructura 1-5)
    'Árbol de Decisión': (5, 1), 'Regresión Logística': (5, 1), 'Random Forest': (2, 4),
    'XGBoost': (2, 2), 'Votación RF+XGB': (1, 5),
}
filas = []
for n in FINALES:
    filas.append({
        'Modelo': n,
        'Recall_CV':    cv_audit.loc[n, 'Recall_media'],
        'Precision_CV': 1 - cv_audit.loc[n, 'Waste Rate_media'],
        'Waste_CV':     cv_audit.loc[n, 'Waste Rate_media'],
        'PR-AUC_CV':    cv_audit.loc[n, 'PR-AUC_media'],
        'PR-AUC_std':   cv_audit.loc[n, 'PR-AUC_std'],
        'Equidad':      tabla_equidad.loc[n, 'Ratio equidad Nicho/Mainstream (OOF)'],
        'Brecha_PRAUC': bitacora_df.loc[(n, 'PR-AUC'), 'Brecha'],
        't_fit_seg':    t_fit[n],
        'Interpretabilidad (1-5)': cualitativo[n][0],
        'Costo_Infra (1-5)':       cualitativo[n][1],
    })
matriz = pd.DataFrame(filas).set_index('Modelo')

def minmax(s):
    s = s.astype(float)
    return (s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else s * 0 + 1

def puntaje(m, pesos):
    norm = pd.DataFrame({
        'Precisión':         minmax(m['Precision_CV']),
        'PR-AUC':            minmax(m['PR-AUC_CV']),
        'Equidad':           minmax(m['Equidad']),
        'Estabilidad':       minmax(-m['PR-AUC_std']),
        'Generalización':    minmax(-m['Brecha_PRAUC'].clip(lower=0)),
        'Interpretabilidad': minmax(m['Interpretabilidad (1-5)']),
        'Costo':             minmax(-m['Costo_Infra (1-5)']),
    })
    return (norm * pd.Series(pesos)).sum(axis=1), norm

PESOS = {'Precisión': 0.30, 'PR-AUC': 0.25, 'Equidad': 0.15, 'Estabilidad': 0.10,
         'Generalización': 0.10, 'Interpretabilidad': 0.05, 'Costo': 0.05}
assert abs(sum(PESOS.values()) - 1) < 1e-9
matriz['Score_Ponderado'], norm = puntaje(matriz, PESOS)
matriz = matriz.sort_values('Score_Ponderado', ascending=False)
GANADOR = matriz.index[0]
print("=== MATRIZ DE DECISIÓN (valores reales de CV) ===")
display(matriz.round(4))
print(f"🏆 SOLUCIÓN ÓPTIMA (score ponderado): {GANADOR}")

In [ ]:
# Radar multidimensional (valores normalizados 0-1)
cats = list(PESOS)
ang = np.linspace(0, 2 * np.pi, len(cats), endpoint=False).tolist(); ang += ang[:1]
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for n in matriz.index:
    v = norm.loc[n, cats].tolist(); v += v[:1]
    ax.plot(ang, v, lw=2.5 if n == GANADOR else 1.5, color=COLORES[n], label=n)
    ax.fill(ang, v, color=COLORES[n], alpha=0.10 if n == GANADOR else 0.03)
ax.set_xticks(ang[:-1]); ax.set_xticklabels(cats, fontsize=9); ax.set_yticklabels([])
ax.set_title('Comparación multidimensional (normalizada)', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=8)
plt.tight_layout(); guardar_fig('radar_decision'); plt.show()

### 14.2 Análisis de sensibilidad de los pesos
Una decisión robusta no debería depender de un ajuste fino de los pesos. Se recalcula el ganador bajo tres
escenarios alternativos de prioridad del negocio.

In [ ]:
escenarios = {
    'Oficial': PESOS,
    'Foco comercial (Precisión/PR-AUC)': {'Precisión': 0.45, 'PR-AUC': 0.35, 'Equidad': 0.05, 'Estabilidad': 0.05,
                                       'Generalización': 0.05, 'Interpretabilidad': 0.025, 'Costo': 0.025},
    'Foco ético (equidad/interpretabilidad)': {'Precisión': 0.20, 'PR-AUC': 0.15, 'Equidad': 0.30, 'Estabilidad': 0.10,
                                               'Generalización': 0.10, 'Interpretabilidad': 0.10, 'Costo': 0.05},
    'Pesos iguales': {k: 1 / 7 for k in PESOS},
}
sens = pd.DataFrame({esc: puntaje(matriz, p)[0] for esc, p in escenarios.items()})
display(sens.round(3))
ganadores = sens.idxmax()
print("Ganador por escenario:", ganadores.to_dict())
robusto = (ganadores == GANADOR).mean()
print(f"=> {GANADOR} gana en {robusto:.0%} de los escenarios: "
      + ("decisión ROBUSTA a la ponderación." if robusto >= 0.75 else "decisión SENSIBLE a la ponderación; se documenta el trade-off."))

### 14.3 Confirmación en test y párrafo ejecutivo
Recién ahora se mira el test para el modelo elegido: es la estimación honesta de su desempeño en producción.

In [ ]:
r = matriz.loc[GANADOR]; mt = comp_opt.loc[GANADOR]
segundo = matriz.index[1]

def parrafo_ejecutivo():
    texto = (
        f"Se recomienda desplegar {GANADOR} como motor de priorización de lanzamientos. "
        f"En validación cruzada de {K_FOLDS} pliegues alcanza PR-AUC={r['PR-AUC_CV']:.3f} (±{r['PR-AUC_std']:.3f}) "
        f"y, con un umbral de {UMBRAL[GANADOR]:.2f} que garantiza descubrir al menos el 70% de los hits, desperdicia "
        f"{r['Waste_CV']:.1%} del presupuesto frente al {1 - y_train.mean():.0%} de promocionar al azar. En el conjunto de "
        f"prueba (artistas nunca vistos, no usado para decidir) confirma Recall={mt['Recall']:.3f}, Waste Rate="
        f"{mt['Waste Rate']:.3f} y PR-AUC={mt['PR-AUC']:.3f}; para descubrir el 70% de los hits desperdicia "
        f"{WASTE70['Árbol de Decisión'] - WASTE70[GANADOR]:.1%} menos presupuesto que el árbol simple, lo que "
        f"demuestra el aporte del ensamble. Frente a {segundo} obtiene mejor puntaje ponderado "
        f"({r['Score_Ponderado']:.3f} vs {matriz.loc[segundo, 'Score_Ponderado']:.3f}) al integrar desempeño, estabilidad, "
        f"equidad (ratio nicho/mainstream {r['Equidad']:.2f}) y costo. Su principal limitación es que la música por sí "
        f"sola explica poco del éxito y que favorece a los géneros que la plataforma ya premia, por lo que debe usarse "
        f"como señal complementaria, con cupos de diversidad, cuota para artistas emergentes y revisión humana."
    )
    return "\n".join(textwrap.wrap(texto, width=100))

print("=== PÁRRAFO EJECUTIVO PARA LA DIRECCIÓN DEL SELLO ===\n")
print(parrafo_ejecutivo())

print("\n=== CHEQUEO DE EQUILIBRIO ===")
print(f"{'Modelo':22s} {'Precis':>7s} {'PR-AUC':>7s} {'σ':>6s} {'Equidad':>8s} {'Interp':>6s} {'Costo':>6s} {'Score':>6s}")
for m_ in matriz.index:
    x_ = matriz.loc[m_]
    print(f"{m_:22s} {x_['Precision_CV']:7.3f} {x_['PR-AUC_CV']:7.3f} {x_['PR-AUC_std']:6.3f} {x_['Equidad']:8.2f} "
          f"{int(x_['Interpretabilidad (1-5)']):6d} {int(x_['Costo_Infra (1-5)']):6d} {x_['Score_Ponderado']:6.3f}")

### 14.4 Verificación de KPIs del proyecto (`kpis_spotify.csv`)
Se contrasta el modelo elegido en **test** contra las metas definidas en EV1. Los KPIs de predicción de valor (MAE,
RMSE, R²) provienen de la ablación de regresión (Sección 10.3).
El **Diversity Score** se mide sobre la **lista final que se promocionaría**, es decir, después del re-ranking con
cupo de 10% por género (Sección 13.4), porque ese re-ranking forma parte de la solución recomendada.

In [ ]:
reg_full = ablacion.loc[ETQ['lanzamiento']]
mt = comp_opt.loc[GANADOR]
metas = {  # (valor, meta, regla de cumplimiento)  — metas tomadas de kpis_spotify.csv
    'MAE':             (reg_full['MAE (popularidad)'],  '≤ 10 puntos',  reg_full['MAE (popularidad)'] <= 10),
    'RMSE':            (reg_full['RMSE (popularidad)'], '< 12 puntos',  reg_full['RMSE (popularidad)'] < 12),
    'R2':              (R2_MUSICA, '≥ 0.45 (solo música)', R2_MUSICA >= 0.45),
    'Recall@Hit':      (mt['Recall'],      '≥ 0.70', mt['Recall'] >= 0.70),
    'PR-AUC':          (mt['PR-AUC'],      '> 0.50', mt['PR-AUC'] > 0.50),
    'F2-Score':        (mt['F2'],          '> 0.65', mt['F2'] > 0.65),
    'NDCG@20':         (tabla_rank.loc[GANADOR, 'NDCG@20 (media por género)'], '> 0.80',
                        tabla_rank.loc[GANADOR, 'NDCG@20 (media por género)'] > 0.80),
    'Waste Rate':      (mt['Waste Rate'],  '≤ 0.15', mt['Waste Rate'] <= 0.15),
    'Diversity Score': (tabla_rerank.loc[GANADOR, 'Géneros con cupo'], '≥ 15 géneros (lista final con cupo 10%)',
                        tabla_rerank.loc[GANADOR, 'Géneros con cupo'] >= 15),
}
kpi_final = kpis[['KPI', 'Meta Sugerida']].copy()
kpi_final['Valor (test)'] = kpi_final['KPI'].map(lambda k: round(float(metas[k][0]), 4))
kpi_final['Meta operativa'] = kpi_final['KPI'].map(lambda k: metas[k][1])
kpi_final['Estado'] = kpi_final['KPI'].map(lambda k: '✅ CUMPLE' if metas[k][2] else '❌ NO CUMPLE')
print(f"Modelo evaluado: {GANADOR} | umbral {UMBRAL[GANADOR]:.2f}")
display(kpi_final)

n_ok = (kpi_final['Estado'] == '✅ CUMPLE').sum()
print(f"KPIs cumplidos: {n_ok}/{len(kpi_final)}")
for _, row in kpi_final[kpi_final['Estado'] != '✅ CUMPLE'].iterrows():
    k = row['KPI']
    motivo = {
        'R2': 'la música sola no explica la popularidad: confirma el sesgo de retroalimentación (hallazgo, no fallo técnico).',
        'Recall@Hit': 'el umbral OOF no se transfirió exactamente al test; bajar levemente el umbral recupera el 70%.',
        'F2-Score': 'consecuencia del umbral alto que protege el presupuesto (F2 premia recall).',
        'NDCG@20': 'el ranking dentro de géneros con pocos hits es poco informativo.',
        'Diversity Score': (f"el top-200 se concentra en géneros mainstream; con el re-ranking de cupo 10% sube a "
                            f"{tabla_rerank.loc[GANADOR, 'Géneros con cupo']} géneros (precisión "
                            f"{tabla_rerank.loc[GANADOR, 'Precisión top-200 original']:.2f} -> {tabla_rerank.loc[GANADOR, 'Precisión top-200 con cupo 10%']:.2f})."),
        'Waste Rate': 'meta fijada con el modelo que tenía fuga; con artistas nuevos, Waste ≤ 15% implica un Recall muy bajo (Sección 9.1). '
                      'Se recomienda redefinir la meta respecto del azar (1 - prevalencia).',
        'Waste Rate (antiguo)': 'el umbral OOF no se transfirió exactamente al test; recalibrar con más datos.',
        'MAE': 'la popularidad tiene alta dispersión dentro de perfiles similares.',
        'RMSE': 'errores grandes en canciones de artistas sin historial (cold-start).',
    }.get(k, 'revisar en la siguiente iteración.')
    print(f"  - {k}: {motivo}")

## 15. Conclusiones, limitaciones y recomendaciones

La celda siguiente genera las conclusiones con los valores reales de esta ejecución, para que el informe
Markdown (README) y la presentación citen cifras consistentes con el notebook.

In [ ]:
c = []
c.append(f"1. OPTIMIZACIÓN (IE9): Δ PR-AUC en CV (optimizado - base EV2) = "
         f"{tabla_ie9.loc['XGBoost', 'Δ (opt - base)']:+.4f} en XGB ({'optimizado' if usa_opt_xgb else 'se conservó base'}) y "
         f"{tabla_ie9.loc['Random Forest', 'Δ (opt - base)']:+.4f} en RF ({'optimizado' if usa_opt_rf else 'se conservó base'})."
         f" La mayor ganancia de negocio vino de optimizar el umbral: pasó de 0.50 a {UMBRAL[GANADOR]:.2f}, reduciendo el Waste Rate "
         f"de {comp_05.loc[GANADOR, 'Waste Rate']:.1%} a {comp_opt.loc[GANADOR, 'Waste Rate']:.1%} en test.")
c.append(f"2. ENSAMBLES (IE10): para descubrir el 70% de los hits (curva PR del test), el desperdicio es {WASTE70['Random Forest']:.1%} (RF), "
         f"{WASTE70['XGBoost']:.1%} (XGB) y {WASTE70['Votación RF+XGB']:.1%} (votación), frente a "
         f"{WASTE70['Árbol de Decisión']:.1%} del árbol simple y {1 - y_test.mean():.1%} del azar."
         + (" El umbral del árbol, además, no se transfiere del CV al test (probabilidades escalonadas)."
            if comp_opt.loc['Árbol de Decisión', 'Recall'] < RECALL_MIN - 0.10 else ""))
c.append(f"3. GENERALIZACIÓN (IE11): σ del PR-AUC en CV K=5 entre {cv_audit['PR-AUC_std'].min():.4f} y "
         f"{cv_audit['PR-AUC_std'].max():.4f} ("
         + ("todas ≤ 0.05 => estables" if cv_audit['PR-AUC_std'].max() <= 0.05 else "alguna > 0.05 => revisar estabilidad")
         + f"). El test confirma la CV del ganador "
         f"(PR-AUC CV {matriz.loc[GANADOR, 'PR-AUC_CV']:.3f} vs test {comp_opt.loc[GANADOR, 'PR-AUC']:.3f}). "
         f"Una CV aleatoria habría inflado el PR-AUC en {tabla_fuga['Inflación PR-AUC'].iloc[-1]:+.3f}.")
c.append(f"4. SELECCIÓN (IE12): {GANADOR} obtiene el mayor puntaje ponderado ({matriz.loc[GANADOR, 'Score_Ponderado']:.3f}) "
         f"y gana en {robusto:.0%} de los escenarios de sensibilidad"
         + (" (decisión robusta)." if robusto >= 0.75 else " (trade-off: con foco ético ganan modelos más simples).")
         + f" Sobreajuste del ganador: brecha PR-AUC train-CV = {matriz.loc[GANADOR, 'Brecha_PRAUC']:.3f}.")
c.append(f"5. SESGOS: la música sola explica R²={R2_MUSICA:.2f} de la popularidad vs "
         f"{R2_LANZ:.2f} con género y catálogo; ratio de equidad nicho/mainstream = "
         f"{tabla_equidad.loc[GANADOR, 'Ratio equidad Nicho/Mainstream (test)']:.2f}.")
print("\n\n".join("\n".join(textwrap.wrap(x, 105)) for x in c))

### Limitaciones
- **Ausencia de variables de contexto**: seguidores, sello, presencia en playlists editoriales, fecha de lanzamiento.
  Por eso la identidad del artista actúa como *proxy* de marketing.
- **La popularidad es una métrica de la plataforma** basada en reproducciones recientes (sesgo de recencia) y no
  mide calidad musical.
- **Muestreo del dataset**: ~1.000 canciones por género obtenidas vía API; no representa la distribución real del
  catálogo de Spotify.
- **Artistas emergentes**: menor recall para artistas con catálogo pequeño (Sección 13.2).
- **Géneros de nicho** con 0–3 positivos en test no permiten métricas individuales estables (se agruparon en terciles).
- La búsqueda de hiperparámetros se hizo sobre una submuestra y con un número acotado de iteraciones por costo computacional.

### Recomendaciones para el negocio
1. **Desplegar el modelo elegido como señal de priorización**, no como filtro excluyente; la decisión final de
   inversión debe incluir criterio humano (A&R).
2. **Operar con el umbral de negocio documentado** (Recall ≥ 70% con mínimo Waste) y revisarlo si cambia el
   presupuesto: la curva PR permite recalcularlo sin reentrenar. **Redefinir la meta de Waste Rate** (15% se fijó
   con un modelo con fuga) como mejora respecto del azar; la tabla de políticas (Sección 9.1) muestra el costo de
   cada alternativa: es una decisión de negocio, no técnica.
3. **Cuota protegida (≈20%) del presupuesto para artistas sin historial**, evaluados con la variante acústica+género,
   para cortar el círculo de retroalimentación.
4. **Monitorear en cada iteración** el ratio de equidad nicho/mainstream, el NDCG@20 y la diversidad del top-200; si
   la brecha crece, activar revisión de sesgo.
5. **Reentrenamiento periódico** con datos que incluyan las intervenciones promocionales (para separar efecto
   marketing de efecto música) y vigilancia de *stream farming*.
6. **Líneas futuras**: stacking con meta-modelo calibrado, embeddings de audio, umbrales por grupo de género y
   early stopping en XGBoost.

## 16. Descubrimientos, insights y resumen para la defensa

Esta sección resume, con las cifras de esta ejecución, lo que el proyecto **descubrió**, **por qué se eligió** el
modelo, **qué significan** sus métricas, **qué sesgos** tiene, **qué hicimos** para mitigarlos, **cómo impactan** y
**qué precauciones** tomar al usarlo. Está pensada como guion para la presentación y para el README.

### 16.1 Descubrimientos e insights

In [ ]:
bG = barridos[GANADOR]
rec_max_15 = bG.loc[bG['Waste Rate'] <= WASTE_MAX, 'Recall'].max()
waste_min_70 = bG.loc[bG['Recall'] >= RECALL_MIN, 'Waste Rate'].min()
blq = imp_bloques['XGBoost'].set_index('Bloque')['% de la caída total']
ta = tabla_artistas.loc[GANADOR]
insights = [
    ("La evaluación anterior era optimista",
     f"Con CV aleatoria el PR-AUC de XGBoost sube {tabla_fuga.loc['XGBoost', 'Inflación PR-AUC']:+.3f}; el TE de álbum "
     f"agrega {ablacion.loc[ETQ['con_te'], 'PR-AUC (clasif.)'] - ablacion.loc[ETQ['lanzamiento'], 'PR-AUC (clasif.)']:+.3f} "
     "incluso con artistas nuevos: en los álbumes compartidos (compilados, reediciones) la popularidad es homogénea, "
     "así que la media del álbum adivina la etiqueta. "
     f"Además hay {df.duplicated(['track_name', 'artists']).sum():,} canciones repetidas con otro track_id."),
    ("La música sola explica poco",
     f"Solo audio: R² = {R2_MUSICA:.2f} y PR-AUC = {ablacion.loc[ETQ['musica'], 'PR-AUC (clasif.)']:.3f} "
     f"(azar = {y_test.mean():.3f}). Con el género: R² = {R2_LANZ:.2f} y PR-AUC = "
     f"{ablacion.loc[ETQ['lanzamiento'], 'PR-AUC (clasif.)']:.3f}."),
    ("El mercado pesa más que el sonido",
     f"Al permutar bloques, Género explica {blq.filter(like='Género').sum():.0f}% de la importancia, Acústicas "
     f"{blq.filter(like='Acústicas').sum():.0f}% e Ingeniería {blq.filter(like='Ingeniería').sum():.0f}%."),
    ("Recall y desperdicio están en conflicto",
     f"Con Waste ≤ 15% el máximo Recall alcanzable es {(rec_max_15 if pd.notna(rec_max_15) else 0):.2f}; para llegar a Recall 70% "
     f"hay que aceptar un Waste de {waste_min_70:.2f} (al azar sería {1 - y_train.mean():.2f}). La meta de 15% venía del "
     "modelo con fuga: la política oficial prioriza el Recall y minimiza el desperdicio."),
    ("La mayor ganancia vino del umbral, no de los hiperparámetros",
     f"La búsqueda mejoró el PR-AUC en CV en {tabla_ie9.loc['XGBoost', 'Δ (opt - base)']:+.4f} (XGB); el umbral redujo "
     f"el Waste de {comp_05.loc[GANADOR, 'Waste Rate']:.1%} a {comp_opt.loc[GANADOR, 'Waste Rate']:.1%}."),
    ("Los ensambles sí aportan",
     f"Para descubrir el 70% de los hits, {GANADOR} desperdicia {WASTE70[GANADOR]:.1%} del presupuesto vs "
     f"{WASTE70['Árbol de Decisión']:.1%} del árbol simple, {WASTE70['Regresión Logística']:.1%} de la logística y "
     f"{1 - y_test.mean():.1%} del azar."
     + (" El árbol además es inestable: su umbral no se transfiere al test."
        if comp_opt.loc['Árbol de Decisión', 'Recall'] < RECALL_MIN - 0.10 else
        f" El árbol además es inestable entre pliegues (σ Recall = {cv_audit.loc['Árbol de Decisión', 'Recall_std']:.3f}).")),
    ("El modelo es estable",
     f"σ del PR-AUC entre pliegues ≤ {cv_audit['PR-AUC_std'].max():.3f}; CV y test difieren en "
     f"{comp_opt.loc[GANADOR, 'PR-AUC'] - matriz.loc[GANADOR, 'PR-AUC_CV']:+.3f}."),
    ("El sesgo está en el género, no en el tamaño del artista",
     f"Recall en géneros de nicho / mainstream = {tabla_equidad.loc[GANADOR, 'Ratio equidad Nicho/Mainstream (test)']:.2f}; "
     f"en cambio, artistas de 1 canción {ta.loc['1 canción', 'Recall']:.2f} vs >10 canciones {ta.loc['>10 canciones', 'Recall']:.2f} "
     f"(tasa real de hit {ta.loc['1 canción', 'Tasa hit real']:.2f} vs {ta.loc['>10 canciones', 'Tasa hit real']:.2f}): "
     "la hipótesis de que el modelo subestima a emergentes no se confirma en estos datos."),
    ("Las probabilidades no son probabilidades reales",
     f"La probabilidad media predicha ({proba_test[GANADOR].mean():.2f}) supera la tasa real de hit ({y_test.mean():.2f}) "
     "porque scale_pos_weight infla la clase hit: sirven para ordenar y decidir con umbral, no para decir 'tiene 70% de chances'."),
    ("Una métrica de train puede mentir",
     "Durante la verificación, el Target Encoding hacía que hasta la regresión logística mostrara PR-AUC de train ≈ 0.99 "
     "(usaba la etiqueta de la propia canción). Medir el train sobre la codificación cruzada corrigió el diagnóstico."),
]
for i, (titulo, texto) in enumerate(insights, 1):
    print(f"{i}. {titulo.upper()}\n   " + "\n   ".join(textwrap.wrap(texto, 100)) + "\n")

### 16.2 Por qué se eligió el modelo

In [ ]:
rG = matriz.loc[GANADOR]
otros = [m_ for m_ in matriz.index if m_ != GANADOR]
razones = [
    f"Mayor puntaje en la matriz de decisión ({rG['Score_Ponderado']:.3f}) construida SOLO con métricas de CV agrupada "
    f"por artista, sin mirar el test (evita sesgo de selección).",
    f"Mejor equilibrio entre costo (Waste CV {rG['Waste_CV']:.3f} con Recall ≥ 70%) y calidad del ranking "
    f"(PR-AUC CV {rG['PR-AUC_CV']:.3f}).",
    (f"Decisión robusta: gana en {robusto:.0%} de los escenarios de pesos (comercial, ético, pesos iguales)."
     if robusto >= 0.75 else
     f"Gana en {robusto:.0%} de los escenarios de pesos: es la mejor opción cuando se prioriza desempeño, pero con foco "
     f"ético/interpretabilidad ganan modelos más simples ({', '.join(sorted(set(ganadores[ganadores != GANADOR])))}). "
     "Se documenta como trade-off explícito."),
    f"Diferencia con {m2 if GANADOR == m1 else m1}: IC95% de la diferencia de PR-AUC = [{ic[0]:+.4f}, {ic[1]:+.4f}] "
    + ("(estadísticamente distinta de 0)." if (ic[0] > 0 or ic[1] < 0) else "(empate estadístico: decidieron estabilidad, costo y equidad)."),
    f"Generaliza: σ PR-AUC {rG['PR-AUC_std']:.3f} entre pliegues y brecha Train-CV {rG['Brecha_PRAUC']:+.3f}.",
    f"Costo e interpretabilidad aceptables frente a la votación (que entrena dos modelos) y con SHAP/importancias "
    "disponibles para explicar decisiones.",
]
print(f"MODELO ELEGIDO: {GANADOR} (umbral {UMBRAL[GANADOR]:.2f})\n")
for r_ in razones:
    print("- " + "\n  ".join(textwrap.wrap(r_, 100)))
print("\nDescartados: " + "; ".join(f"{m_} ({matriz.loc[m_, 'Score_Ponderado']:.3f})" for m_ in otros))

### 16.3 Qué representan las métricas (explicación para negocio)

In [ ]:
mG = comp_opt.loc[GANADOR]
explic = pd.DataFrame([
    ('Recall@Hit', mG['Recall'], 'De cada 100 hits reales, cuántos detecta el modelo.',
     f"Descubre {mG['Recall'] * 100:.0f} de cada 100 éxitos; el resto es oportunidad perdida."),
    ('Precision', mG['Precision'], 'De cada 100 canciones que el modelo manda a promocionar, cuántas son hit.',
     f"{mG['Precision'] * 100:.0f} de cada 100 promociones aciertan."),
    ('Waste Rate', mG['Waste Rate'], '1 − Precision: fracción del presupuesto invertida en canciones que no serán hit.',
     f"Se desperdicia {mG['Waste Rate']:.0%} del presupuesto (meta ≤ 15%)."),
    ('F2', mG['F2'], 'Promedio de precision y recall que pesa el doble el recall.', 'Resume el balance priorizando no perder hits.'),
    ('PR-AUC', mG['PR-AUC'], 'Calidad del ranking para la clase hit, para cualquier umbral. Azar = prevalencia.',
     f"{mG['PR-AUC'] / y_test.mean():.1f} veces mejor que elegir al azar ({y_test.mean():.2f})."),
    ('ROC-AUC', mG['ROC-AUC'], 'Probabilidad de que un hit reciba mayor puntaje que un no-hit.',
     f"En {mG['ROC-AUC']:.0%} de los pares hit/no-hit el modelo los ordena bien."),
    ('Brier', mG['Brier'], 'Error cuadrático de las probabilidades (calibración). Menor es mejor.',
     'Indica si "70% de probabilidad" significa realmente ~70%.'),
    ('NDCG@20', tabla_rank.loc[GANADOR, 'NDCG@20 (media por género)'], 'Calidad del orden de las 20 primeras canciones de cada género.',
     'Útil para armar playlists ordenadas.'),
    ('Diversity (top-200)', tabla_rank.loc[GANADOR, 'Géneros en top-200'], 'Géneros distintos entre las 200 canciones mejor puntuadas.',
     'Controla que la promoción no se concentre en pocos géneros.'),
    ('σ entre pliegues', matriz.loc[GANADOR, 'PR-AUC_std'], 'Variación del PR-AUC entre particiones de CV.',
     '≤ 0.05: el desempeño no depende de la muestra.'),
], columns=['Métrica', 'Valor', 'Qué representa', 'Lectura para el negocio']).set_index('Métrica')
display(explic.round(3))

### 16.4 Sesgos del modelo: qué hicimos, cómo impactan y qué precauciones tomar

In [ ]:
tabla_sesgos = pd.DataFrame([
    {'Sesgo': 'Fuga de información',
     'Qué hicimos': 'Split y CV por artista; eliminamos TE de álbum/artista; umbral con OOF; test una sola vez',
     'Impacto si no se controla': f"Métricas infladas (≈{tabla_fuga.loc['XGBoost', 'Inflación PR-AUC']:+.2f} PR-AUC) y presupuesto decidido con un modelo peor de lo que se cree",
     'Precaución en producción': 'Validar siempre con artistas/períodos no vistos; nunca usar datos posteriores al lanzamiento'},
    {'Sesgo': 'Efecto Mateo (retroalimentación)',
     'Qué hicimos': 'Medimos cuánto explica la música sola (ablación) y cuánto el contexto; lo declaramos',
     'Impacto si no se controla': 'El modelo promociona lo que ya era popular; el éxito se vuelve profecía autocumplida',
     'Precaución en producción': 'Usar como apoyo, no como filtro; cuota de presupuesto para artistas sin historial'},
    {'Sesgo': 'Artistas emergentes',
     'Qué hicimos': 'Sin TE de artista ni catálogo como variables; recall medido por tamaño de catálogo',
     'Impacto si no se controla': f"Hoy: recall {ta.loc['1 canción', 'Recall']:.2f} (1 canción) vs {ta.loc['>10 canciones', 'Recall']:.2f} (>10). "
                                  "Si se agregara historial del artista, el modelo premiaría a los ya establecidos",
     'Precaución en producción': 'Umbral o cupo específico por segmento; revisión humana (A&R) de los descartados'},
    {'Sesgo': 'Homogeneización cultural (géneros)',
     'Qué hicimos': 'Equidad nicho/mainstream como criterio de la matriz de decisión; diversidad del top-200',
     'Impacto si no se controla': f"Recall de nicho = {tabla_equidad.loc[GANADOR, 'Ratio equidad Nicho/Mainstream (test)']:.0%} del mainstream; catálogo cada vez más uniforme",
     'Precaución en producción': f"Re-ranking con cupo por género (Sección 13.4: {tabla_rerank.loc[GANADOR, 'Géneros original']} -> "
                                 f"{tabla_rerank.loc[GANADOR, 'Géneros con cupo']} géneros en el top-200); reportar el ratio en cada iteración"},
    {'Sesgo': 'Conflicto de KPIs',
     'Qué hicimos': 'Política explícita: Recall ≥ 70% con mínimo Waste; tabla con el costo de cada política alternativa',
     'Impacto si no se controla': f"Con umbral 0.50 se desperdiciaría {comp_05.loc[GANADOR, 'Waste Rate']:.0%} del presupuesto",
     'Precaución en producción': 'Revisar el umbral cuando cambie el presupuesto; documentarlo como decisión de negocio'},
    {'Sesgo': 'Recencia, muestreo y manipulación',
     'Qué hicimos': 'Declarado como limitación (no medible con estos datos)',
     'Impacto si no se controla': 'Catálogo histórico subvalorado; stream farming contamina el target',
     'Precaución en producción': 'Agregar fecha de lanzamiento; reentrenar periódicamente; auditar anomalías de streams'},
]).set_index('Sesgo')
display(tabla_sesgos)

### 16.5 Checklist de uso responsable
1. **Señal, no veredicto**: la probabilidad ordena la lista; la decisión de invertir la toma una persona.
2. **Umbral documentado** (Recall ≥ 70% con mínimo Waste) y recalculado si cambia el presupuesto (no requiere reentrenar).
3. **Cuota protegida** (≈20%) para artistas sin historial y géneros de nicho.
4. **Monitoreo por segmento** en cada iteración: recall por género, por tamaño de catálogo y diversidad del top-200.
5. **Sin variables posteriores al lanzamiento** (popularidad del álbum, playlists) aunque "mejoren" las métricas.
6. **Reentrenamiento periódico** y vigilancia de *stream farming*.
7. **Transparencia**: explicar a los artistas qué variables pesan (Sección 10) si una decisión los afecta.

## 17. Exportación de artefactos (reproducibilidad)
Se guardan el pipeline ganador (incluye el preprocesamiento completo), el umbral oficial, las tablas de resultados
y las predicciones del test, siguiendo la estructura `data/ · notebooks/ · models/ · images/ · reports/`.

> Para cargar el modelo en otra sesión, ejecute antes la celda de la Sección 4 (define `VotacionSuave`, necesaria
> para deserializar si el ganador es la votación).

In [ ]:
artefacto = {'modelo': modelos[GANADOR], 'umbral': UMBRAL[GANADOR], 'politica': POLITICA,
             'features_entrada': FEATURES_LANZAMIENTO, 'umbral_hit': UMBRAL_HIT,
             'versiones': {'sklearn': sklearn.__version__, 'xgboost': xgboost.__version__}}
ruta_modelo = os.path.join(DIRS['models'], 'ev3_modelo_final.joblib')
joblib.dump(artefacto, ruta_modelo, compress=3)

tablas = {'ev3_busqueda_hiperparametros.csv': tabla_ie9, 'ev3_comparacion_umbral05.csv': comp_05,
          'ev3_comparacion_umbral_optimo.csv': comp_opt.assign(**{'Waste a Recall 70% (curva PR)': pd.Series(WASTE70)}), 'ev3_politicas_umbral.csv': tabla_pol,
          'ev3_cv_k5.csv': cv_audit, 'ev3_train_vs_cv.csv': bitacora_df, 'ev3_fuga_cv_aleatoria.csv': tabla_fuga, 'ev3_artistas.csv': tabla_artistas,
          'ev3_ablacion.csv': ablacion, 'ev3_importancia_bloques.csv': tabla_bloques,
          'ev3_equidad.csv': tabla_equidad, 'ev3_ranking_diversidad.csv': tabla_rank, 'ev3_rerank_diversidad.csv': tabla_rerank,
          'ev3_matriz_decision.csv': matriz, 'ev3_sensibilidad_pesos.csv': sens, 'ev3_kpis.csv': kpi_final}
for nombre, t in tablas.items():
    t.to_csv(os.path.join(DIRS['reports'], nombre))

pred = df_test[['track_id', 'track_name', 'artists', 'artist_n_tracks', 'genre', 'popularity', 'is_hit']].copy()
pred['prob_hit'] = proba_test[GANADOR]
pred['pred_hit'] = (pred['prob_hit'] >= UMBRAL[GANADOR]).astype(int)
pred['popularidad_estimada'] = modelos_reg[ETQ['lanzamiento']][1]
pred.sort_values('prob_hit', ascending=False).to_csv(os.path.join(DIRS['reports'], 'ev3_predicciones_test.csv'), index=False)

print(f"Modelo guardado en: {ruta_modelo}")
print(f"Tablas: {len(tablas)} CSV en {DIRS['reports']} | Figuras en {DIRS['images']}")

# Prueba de carga (reproducibilidad)
cargado = joblib.load(ruta_modelo)
p_chk = cargado['modelo'].predict_proba(X_test.head(5))[:, 1]
assert np.allclose(p_chk, proba_test[GANADOR][:5]), 'El modelo recargado no reproduce las predicciones'
print("✅ El modelo recargado reproduce exactamente las predicciones.")